<a href="https://colab.research.google.com/github/rezahamzeh69/TARL_NET_extended_v3/blob/main/TARL_NET_extended_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TARL-Net v2 — **v3** notebook (final review additions)

This notebook keeps the full pipeline and adds the five reviewer-requested items. Code cells contain **no inline comments** by request.

**Added in v3 (entry point `main_full_v3`, last cell):**
1. **Independent-dataset zero-shot transfer** — source-trained model applied with no retraining to a shifted/proxy external set; reports attack-AUROC, recall@τ, FPR@τ (honest cross-distribution check). Swap in a real CICIDS2017/ToN-IoT loader where marked.
2. **Multi-seed ablation** — `run_ablation_multiseed` (3 seeds, mean±std) settles the SSL/φ contribution.
3. **MCC + Cohen's κ + balanced-accuracy** reported alongside macro-F1 (`report_with_mcc`) — fairer under severe imbalance.
4. **Continual-learning claim guard** — prints a CONDITIONAL note; abstract must say *mitigates*, not *prevents*, forgetting.
5. **Horizon → wall-clock lead time** (`horizon_to_time_report`) and **multi-horizon prediction** H=1,2,4 (`multi_horizon_prediction`) — strengthens the prediction claim.

**Thesis figure exports** (`emit_thesis_figures`): confusion matrix, full threshold sweep, and pre-train/supervised loss curves are printed as copy-paste JSON blocks.

**Run order:** cell 1 → cell 2 → cell 3 → cell 4 (define) → cell 5 runs `main_full_v3()`. Quick pass: `main_full_v3(ConfigX(seeds=(1337,)))`.

In [1]:
import math
import copy
import os
import re
import glob
import sys
import subprocess
import random
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report,
)


SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


CLASS_NAMES = ["Normal", "DoS", "DDoS", "Probe", "Botnet",
               "BruteForce", "WebAttack", "Infiltration"]
NORMAL_LABEL = 0
ATTACK_FAMILY = {
    "Normal": "benign",
    "DoS": "volumetric",
    "DDoS": "volumetric",
    "Probe": "recon",
    "Botnet": "c2",
    "BruteForce": "credential",
    "WebAttack": "webapp",
    "Infiltration": "intrusion",
}
PROTO_NAMES = ["TCP", "UDP", "ICMP", "OTHER"]


def set_taxonomy(names, families, normal_label=0):
    global CLASS_NAMES, ATTACK_FAMILY, NORMAL_LABEL
    CLASS_NAMES = list(names)
    ATTACK_FAMILY = dict(families)
    NORMAL_LABEL = normal_label


@dataclass
class Config:
    dataset: str = "unsw"
    unsw_slug: str = "mrwellsdavid/unsw-nb15"
    unsw_max_rows: int = 0
    max_windows: int = 80000
    balance_majority: bool = True
    majority_multiple: float = 3.0
    minority_floor: int = 300

    n_hosts: int = 220
    n_services: int = 6
    flows_per_host: int = 90
    raw_dim: int = 40
    nan_ratio: float = 0.01
    mi_keep: int = 26
    corr_thresh: float = 0.96
    seq_len: int = 16
    seq_stride: int = 16
    n_port_buckets: int = 16
    n_protocols: int = 16

    d_model: int = 64
    n_heads: int = 4
    n_transformer_layers: int = 2
    ff_dim: int = 128
    n_lstm_layers: int = 2
    lstm_hidden: int = 64
    dropout: float = 0.1
    svdd_dim: int = 64
    proj_dim: int = 64
    ctx_emb_dim: int = 32

    batch_size: int = 128
    pretrain_epochs: int = 6
    supervised_epochs: int = 12
    lr: float = 1.0e-3
    weight_decay: float = 1.0e-4
    mask_ratio: float = 0.15
    curriculum_warmup: int = 4
    ntxent_temp: float = 0.5
    lambda_mtm: float = 1.0
    lambda_contrast: float = 0.5
    lambda_svdd: float = 0.1
    focal_gamma: float = 2.0
    svdd_quantile: float = 0.95

    ewc_lambda: float = 30.0
    replay_capacity: int = 2000
    replay_per_batch: int = 32

    use_transformer: bool = True
    use_lstm: bool = True
    use_phi: bool = True
    use_ssl: bool = True
    vector_gate: bool = True


def _class_profile(num_classes, raw_dim):
    rng = np.random.RandomState(7)
    means = rng.uniform(-1.0, 1.0, size=(num_classes, raw_dim))
    scales = rng.uniform(0.4, 1.2, size=(num_classes, raw_dim))
    for c in range(num_classes):
        active = rng.choice(raw_dim, size=raw_dim // 3, replace=False)
        means[c, active] += rng.uniform(1.5, 4.0, size=active.shape) * rng.choice([-1, 1], size=active.shape)
    return means, scales


def _port_distribution(num_classes, n_buckets):
    rng = np.random.RandomState(11)
    dist = rng.uniform(0.2, 1.0, size=(num_classes, n_buckets))
    for c in range(num_classes):
        peak = rng.choice(n_buckets, size=2, replace=False)
        dist[c, peak] += rng.uniform(3.0, 6.0, size=peak.shape)
    return dist / dist.sum(axis=1, keepdims=True)


def _proto_distribution(num_classes, n_proto):
    rng = np.random.RandomState(13)
    dist = rng.uniform(0.2, 1.0, size=(num_classes, n_proto))
    return dist / dist.sum(axis=1, keepdims=True)


def generate_flows(cfg):
    num_classes = len(CLASS_NAMES)
    means, scales = _class_profile(num_classes, cfg.raw_dim)
    port_dist = _port_distribution(num_classes, cfg.n_port_buckets)
    proto_dist = _proto_distribution(num_classes, cfg.n_protocols)
    rng = np.random.RandomState(SEED)

    feats, labels, hosts, services, ports, protos, times = [], [], [], [], [], [], []
    for h in range(cfg.n_hosts):
        svc = rng.randint(0, cfg.n_services)
        n = cfg.flows_per_host
        prev = np.zeros(cfg.raw_dim)
        t = 0.0
        i = 0
        while i < n:
            if rng.rand() < 0.30:
                cls = rng.randint(1, num_classes)
                ep_len = rng.randint(4, 14)
            else:
                cls = NORMAL_LABEL
                ep_len = rng.randint(6, 18)
            for _ in range(ep_len):
                if i >= n:
                    break
                noise = rng.randn(cfg.raw_dim) * scales[cls]
                x = 0.6 * prev + 0.4 * (means[cls] + noise)
                prev = x
                pb = rng.choice(cfg.n_port_buckets, p=port_dist[cls])
                pr = rng.choice(cfg.n_protocols, p=proto_dist[cls])
                t += rng.exponential(1.0)
                feats.append(x)
                labels.append(cls)
                hosts.append(h)
                services.append(svc)
                ports.append(pb)
                protos.append(pr)
                times.append(t)
                i += 1

    X = np.asarray(feats, dtype=np.float64)
    y = np.asarray(labels, dtype=np.int64)
    host = np.asarray(hosts, dtype=np.int64)
    service = np.asarray(services, dtype=np.int64)
    port = np.asarray(ports, dtype=np.int64)
    proto = np.asarray(protos, dtype=np.int64)
    ts = np.asarray(times, dtype=np.float64)

    n_nan = int(cfg.nan_ratio * X.size)
    ri = rng.randint(0, X.shape[0], size=n_nan)
    ci = rng.randint(0, X.shape[1], size=n_nan)
    X[ri, ci] = np.nan
    return dict(X=X, y=y, host=host, service=service, port=port, proto=proto, ts=ts)


def impute_and_clip(X, lo=None, hi=None, med=None):
    if med is None:
        med = np.nanmedian(X, axis=0)
    inds = np.where(np.isnan(X))
    X = X.copy()
    X[inds] = np.take(med, inds[1])
    if lo is None or hi is None:
        lo = np.percentile(X, 1, axis=0)
        hi = np.percentile(X, 99, axis=0)
    X = np.clip(X, lo, hi)
    return X, lo, hi, med


def select_features(X, y, keep, corr_thresh):
    mi = mutual_info_classif(X, y, random_state=SEED)
    order = np.argsort(mi)[::-1]
    chosen = []
    for idx in order:
        if len(chosen) >= keep:
            break
        ok = True
        for c in chosen:
            cc = np.corrcoef(X[:, idx], X[:, c])[0, 1]
            if abs(cc) > corr_thresh:
                ok = False
                break
        if ok:
            chosen.append(idx)
    return np.asarray(sorted(chosen), dtype=np.int64)


def build_sequences(data, feat_idx, scaler, cfg, host_set):
    X = data["X"][:, feat_idx]
    X = scaler.transform(X)
    y, host, service = data["y"], data["host"], data["service"]
    port, proto, ts = data["port"], data["proto"], data["ts"]

    seqs, sports, sprotos, slabels = [], [], [], []
    keys = {}
    for i in range(len(y)):
        if host[i] not in host_set:
            continue
        k = (host[i], service[i])
        keys.setdefault(k, []).append(i)

    for k, idxs in keys.items():
        idxs = sorted(idxs, key=lambda j: ts[j])
        if len(idxs) < cfg.seq_len:
            continue
        for s in range(0, len(idxs) - cfg.seq_len + 1, cfg.seq_stride):
            win = idxs[s:s + cfg.seq_len]
            seqs.append(X[win])
            sports.append(port[win])
            sprotos.append(proto[win])
            slabels.append(y[win[-1]])

    Xs = torch.tensor(np.asarray(seqs), dtype=torch.float32)
    Ps = torch.tensor(np.asarray(sports), dtype=torch.long)
    Rs = torch.tensor(np.asarray(sprotos), dtype=torch.long)
    Ys = torch.tensor(np.asarray(slabels), dtype=torch.long)
    return Xs, Ps, Rs, Ys


def make_splits(cfg):
    data = generate_flows(cfg)
    rng = np.random.RandomState(SEED)
    hosts = np.arange(cfg.n_hosts)
    rng.shuffle(hosts)
    n_tr = int(0.6 * cfg.n_hosts)
    n_va = int(0.2 * cfg.n_hosts)
    tr_h = set(hosts[:n_tr].tolist())
    va_h = set(hosts[n_tr:n_tr + n_va].tolist())
    te_h = set(hosts[n_tr + n_va:].tolist())

    Xtr_flat = data["X"][np.isin(data["host"], list(tr_h))]
    ytr_flat = data["y"][np.isin(data["host"], list(tr_h))]
    Xtr_flat, lo, hi, med = impute_and_clip(Xtr_flat)

    feat_idx = select_features(Xtr_flat, ytr_flat, cfg.mi_keep, cfg.corr_thresh)
    scaler = RobustScaler().fit(Xtr_flat[:, feat_idx])

    data["X"], _, _, _ = impute_and_clip(data["X"], lo, hi, med)

    train = build_sequences(data, feat_idx, scaler, cfg, tr_h)
    val = build_sequences(data, feat_idx, scaler, cfg, va_h)
    test = build_sequences(data, feat_idx, scaler, cfg, te_h)
    return train, val, test, feat_idx, scaler


UNSW_COLS = [
    "srcip", "sport", "dstip", "dsport", "proto", "state", "dur", "sbytes", "dbytes",
    "sttl", "dttl", "sloss", "dloss", "service", "sload", "dload", "spkts", "dpkts",
    "swin", "dwin", "stcpb", "dtcpb", "smeansz", "dmeansz", "trans_depth", "res_bdy_len",
    "sjit", "djit", "stime", "ltime", "sintpkt", "dintpkt", "tcprtt", "synack", "ackdat",
    "is_sm_ips_ports", "ct_state_ttl", "ct_flw_http_mthd", "is_ftp_login", "ct_ftp_cmd",
    "ct_srv_src", "ct_srv_dst", "ct_dst_ltm", "ct_src_ltm", "ct_src_dport_ltm",
    "ct_dst_sport_ltm", "ct_dst_src_ltm", "attack_cat", "label",
]
UNSW_NUMERIC = [
    "dur", "sbytes", "dbytes", "sttl", "dttl", "sloss", "dloss", "sload", "dload",
    "spkts", "dpkts", "swin", "dwin", "stcpb", "dtcpb", "smeansz", "dmeansz",
    "trans_depth", "res_bdy_len", "sjit", "djit", "sintpkt", "dintpkt", "tcprtt",
    "synack", "ackdat", "is_sm_ips_ports", "ct_state_ttl", "ct_flw_http_mthd",
    "is_ftp_login", "ct_ftp_cmd", "ct_srv_src", "ct_srv_dst", "ct_dst_ltm",
    "ct_src_ltm", "ct_src_dport_ltm", "ct_dst_sport_ltm", "ct_dst_src_ltm",
]
UNSW_CLASS_NAMES = ["Normal", "Fuzzers", "Analysis", "Backdoor", "DoS", "Exploits",
                    "Generic", "Reconnaissance", "Shellcode", "Worms"]
UNSW_FAMILY = {n: ("benign" if n == "Normal" else n.lower()) for n in UNSW_CLASS_NAMES}
UNSW_ATTACK_ALIASES = {
    "backdoors": "Backdoor", "backdoor": "Backdoor", "fuzzers": "Fuzzers",
    "analysis": "Analysis", "dos": "DoS", "exploits": "Exploits", "generic": "Generic",
    "reconnaissance": "Reconnaissance", "shellcode": "Shellcode", "worms": "Worms",
}


def _ensure_kagglehub():
    try:
        import kagglehub
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
        import kagglehub
    return kagglehub


def _read_unsw_csv(fp):
    head = pd.read_csv(fp, nrows=3, header=None, dtype=str, encoding="latin-1")
    first = [str(v).strip().lower() for v in head.iloc[0].tolist()]
    has_header = "srcip" in first or "proto" in first and "service" in first
    df = pd.read_csv(fp, header=0 if has_header else None, dtype=str,
                     low_memory=False, encoding="latin-1")
    if not has_header:
        n = df.shape[1]
        df.columns = UNSW_COLS[:n] if n <= len(UNSW_COLS) else list(range(n))
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df


def _safe_port(v):
    try:
        return int(float(v))
    except (ValueError, TypeError):
        try:
            return int(str(v).strip(), 16)
        except (ValueError, TypeError):
            return -1


def _bucketize_ports(ports, nb):
    p = np.clip(ports, 0, 65535).astype(np.float64)
    b = np.floor(p / 65536.0 * (nb - 1)).astype(np.int64)
    b[ports < 0] = nb - 1
    return np.clip(b, 0, nb - 1)


def _map_proto(series, nb):
    s = series.astype(str).str.strip().str.lower()
    top = list(s.value_counts().index[: nb - 1])
    mapping = {name: i for i, name in enumerate(top)}
    codes = s.map(mapping).fillna(nb - 1).astype(np.int64).values
    return codes


def load_unsw_nb15(cfg):
    set_taxonomy(UNSW_CLASS_NAMES, UNSW_FAMILY, normal_label=0)
    kagglehub = _ensure_kagglehub()
    path = kagglehub.dataset_download(cfg.unsw_slug)
    all_csv = []
    for root, _, files in os.walk(path):
        for f in files:
            if f.lower().endswith(".csv"):
                all_csv.append(os.path.join(root, f))
    data_files = sorted([f for f in all_csv
                         if re.search(r"nb15_[1-4]\.csv$", os.path.basename(f).lower())])
    if not data_files:
        raise FileNotFoundError(
            "UNSW-NB15_1..4.csv not found under " + path +
            ". Use a kagglehub dataset that contains the full four CSV files.")
    frames = []
    for fp in data_files:
        df = _read_unsw_csv(fp)
        frames.append(df)
        if cfg.unsw_max_rows and sum(len(x) for x in frames) >= cfg.unsw_max_rows:
            break
    df = pd.concat(frames, ignore_index=True)
    if cfg.unsw_max_rows and len(df) > cfg.unsw_max_rows:
        df = df.iloc[: cfg.unsw_max_rows].copy()

    name_to_idx = {n: i for i, n in enumerate(UNSW_CLASS_NAMES)}
    ac = df["attack_cat"].astype(str).str.strip()
    ac = ac.replace({"nan": "", "-": "", "None": ""})
    ac_norm = ac.str.lower().map(UNSW_ATTACK_ALIASES)
    y = ac_norm.map(name_to_idx)
    y = y.where(ac.str.len() > 0, 0).fillna(0).astype(np.int64).values

    Xdf = df[UNSW_NUMERIC].apply(pd.to_numeric, errors="coerce")
    Xdf = Xdf.replace([np.inf, -np.inf], np.nan)
    state_code = pd.Series(pd.factorize(df["state"].astype(str).str.strip())[0],
                           name="state_code")
    X = np.column_stack([Xdf.values.astype(np.float64),
                         state_code.values.astype(np.float64)])

    host = pd.factorize(df["srcip"].astype(str))[0].astype(np.int64)
    service = pd.factorize(df["service"].astype(str).str.strip())[0].astype(np.int64)
    ports = np.array([_safe_port(v) for v in df["dsport"].values], dtype=np.int64)
    port = _bucketize_ports(ports, cfg.n_port_buckets)
    proto = _map_proto(df["proto"], cfg.n_protocols)
    ts = pd.to_numeric(df["stime"], errors="coerce").ffill().fillna(0).values
    if np.allclose(ts, ts[0]):
        ts = np.arange(len(df), dtype=np.float64)

    return dict(X=X, y=y, host=host, service=service, port=port, proto=proto, ts=ts)


def build_window_indices(data, cfg):
    y, host, service, ts = data["y"], data["host"], data["service"], data["ts"]
    groups = {}
    for i in range(len(y)):
        groups.setdefault((int(host[i]), int(service[i])), []).append(i)
    windows = []
    for _, idxs in groups.items():
        idxs.sort(key=lambda j: ts[j])
        for s in range(0, len(idxs) - cfg.seq_len + 1, cfg.seq_stride):
            win = idxs[s:s + cfg.seq_len]
            windows.append((np.asarray(win, dtype=np.int64), int(y[win[-1]])))
    return windows


def stratified_split_windows(windows, cfg):
    rng = np.random.RandomState(SEED)
    by_cls = {}
    for i, (_, lab) in enumerate(windows):
        by_cls.setdefault(lab, []).append(i)
    tr, va, te = [], [], []
    for lab, idxs in by_cls.items():
        idxs = np.asarray(idxs)
        rng.shuffle(idxs)
        n = len(idxs)
        a, b = int(0.6 * n), int(0.8 * n)
        tr += idxs[:a].tolist()
        va += idxs[a:b].tolist()
        te += idxs[b:].tolist()
    if cfg.max_windows and len(tr) > int(0.6 * cfg.max_windows):
        rng.shuffle(tr)
        tr = tr[: int(0.6 * cfg.max_windows)]
        rng.shuffle(va)
        va = va[: int(0.2 * cfg.max_windows)]
        rng.shuffle(te)
        te = te[: int(0.2 * cfg.max_windows)]
    return tr, va, te


def assemble_windows(data, windows, idx_list, feat_idx, scaler):
    Xn = scaler.transform(data["X"][:, feat_idx]).astype(np.float32)
    port, proto = data["port"], data["proto"]
    Xs, Ps, Rs, Ys = [], [], [], []
    for i in idx_list:
        win, lab = windows[i]
        Xs.append(Xn[win])
        Ps.append(port[win])
        Rs.append(proto[win])
        Ys.append(lab)
    return (torch.tensor(np.asarray(Xs), dtype=torch.float32),
            torch.tensor(np.asarray(Ps), dtype=torch.long),
            torch.tensor(np.asarray(Rs), dtype=torch.long),
            torch.tensor(np.asarray(Ys), dtype=torch.long))


def balance_train_indices(idx_list, windows, cfg):
    if not cfg.balance_majority:
        return idx_list
    rng = np.random.RandomState(SEED)
    idx_arr = np.asarray(idx_list)
    labels = np.asarray([windows[i][1] for i in idx_list])
    counts = np.bincount(labels, minlength=len(CLASS_NAMES))
    attack_max = counts[1:].max() if counts[1:].size and counts[1:].max() > 0 else counts.max()
    cap = int(max(attack_max * cfg.majority_multiple, cfg.minority_floor))
    keep = []
    for c in np.unique(labels):
        ci = idx_arr[labels == c]
        if c == NORMAL_LABEL and len(ci) > cap:
            ci = rng.choice(ci, cap, replace=False)
        elif len(ci) < cfg.minority_floor:
            reps = int(np.ceil(cfg.minority_floor / len(ci)))
            ci = np.tile(ci, reps)[: cfg.minority_floor]
        keep.extend(ci.tolist())
    rng.shuffle(keep)
    return keep


def make_splits_unsw(cfg):
    data = load_unsw_nb15(cfg)
    windows = build_window_indices(data, cfg)
    tr, va, te = stratified_split_windows(windows, cfg)
    tr = balance_train_indices(tr, windows, cfg)

    train_flows = np.unique(np.concatenate([windows[i][0] for i in tr]))
    Xtr_flat, lo, hi, med = impute_and_clip(data["X"][train_flows])
    ytr_flat = data["y"][train_flows]
    if len(Xtr_flat) > 60000:
        sub = np.random.RandomState(SEED).choice(len(Xtr_flat), 60000, replace=False)
        feat_idx = select_features(Xtr_flat[sub], ytr_flat[sub], cfg.mi_keep, cfg.corr_thresh)
    else:
        feat_idx = select_features(Xtr_flat, ytr_flat, cfg.mi_keep, cfg.corr_thresh)
    scaler = RobustScaler().fit(Xtr_flat[:, feat_idx])
    data["X"], _, _, _ = impute_and_clip(data["X"], lo, hi, med)

    train = assemble_windows(data, windows, tr, feat_idx, scaler)
    val = assemble_windows(data, windows, va, feat_idx, scaler)
    test = assemble_windows(data, windows, te, feat_idx, scaler)
    return train, val, test, feat_idx, scaler


def temporal_jitter(x, p, r):
    noise = torch.randn_like(x) * 0.08
    perm = x.clone()
    T = x.size(1)
    if T > 1:
        for b in range(x.size(0)):
            i = random.randint(0, T - 2)
            perm[b, i], perm[b, i + 1] = x[b, i + 1].clone(), x[b, i].clone()
    return perm + noise, p, r


def packet_subsample(x, p, r, ratio=0.2):
    T = x.size(1)
    mask = (torch.rand(x.size(0), T, device=x.device) > ratio).float().unsqueeze(-1)
    return x * mask, p, r


def feature_masking(x, p, r, ratio=0.15):
    Fd = x.size(2)
    mask = (torch.rand(x.size(0), 1, Fd, device=x.device) > ratio).float()
    return x * mask, p, r


def augment(x, p, r):
    fns = [temporal_jitter, packet_subsample, feature_masking]
    a, b = random.sample(fns, 2)
    x1, p1, r1 = a(x, p, r)
    x2, p2, r2 = b(x, p, r)
    return (x1, p1, r1), (x2, p2, r2)


class PositionalEncoding(nn.Module):
    def __init__(self, d, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class TransformerEncoderLayerAttn(nn.Module):
    def __init__(self, d, heads, ff, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(dropout), nn.Linear(ff, d))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        a, w = self.attn(x, x, x, need_weights=True, average_attn_weights=True)
        x = self.norm1(x + self.drop(a))
        x = self.norm2(x + self.drop(self.ff(x)))
        return x, w


class TransformerBranch(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.pe = PositionalEncoding(cfg.d_model)
        self.layers = nn.ModuleList(
            [TransformerEncoderLayerAttn(cfg.d_model, cfg.n_heads, cfg.ff_dim, cfg.dropout)
             for _ in range(cfg.n_transformer_layers)]
        )

    def forward(self, x):
        x = self.pe(x)
        attns = []
        for layer in self.layers:
            x, w = layer(x)
            attns.append(w)
        return x, attns


class BiLSTMBranch(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.lstm = nn.LSTM(cfg.d_model, cfg.lstm_hidden, num_layers=cfg.n_lstm_layers,
                            batch_first=True, bidirectional=True,
                            dropout=cfg.dropout if cfg.n_lstm_layers > 1 else 0.0)
        self.proj = nn.Linear(2 * cfg.lstm_hidden, cfg.d_model)

    def forward(self, x):
        h, _ = self.lstm(x)
        return self.proj(h)


class ThreatContextEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.port_emb = nn.Embedding(cfg.n_port_buckets, cfg.ctx_emb_dim)
        self.proto_emb = nn.Embedding(cfg.n_protocols, cfg.ctx_emb_dim)
        self.proj = nn.Sequential(
            nn.Linear(2 * cfg.ctx_emb_dim, cfg.d_model), nn.GELU(), nn.LayerNorm(cfg.d_model)
        )
        self.score = nn.Linear(cfg.d_model, 1)
        self.use_phi = cfg.use_phi

    def forward(self, port, proto):
        c = torch.cat([self.port_emb(port), self.proto_emb(proto)], dim=-1)
        c_pos = self.proj(c)
        phi = c_pos.mean(dim=1)
        ms = self.score(c_pos).squeeze(-1)
        if not self.use_phi:
            phi = torch.zeros_like(phi)
        return phi, c_pos, ms


class VectorAdaptiveGate(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        out = cfg.d_model if cfg.vector_gate else 1
        self.fc = nn.Linear(3 * cfg.d_model, out)
        self.vector = cfg.vector_gate

    def forward(self, ht, hl, phi):
        a = torch.sigmoid(self.fc(torch.cat([ht, hl, phi], dim=-1)))
        return a


class FeatureRefinement(nn.Module):
    def __init__(self, d, dropout):
        super().__init__()
        self.fc = nn.Linear(d, d)
        self.norm = nn.LayerNorm(d)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.norm(x + self.drop(F.gelu(self.fc(x))))


class TARLNetV2(nn.Module):
    def __init__(self, cfg, raw_dim, num_known):
        super().__init__()
        self.cfg = cfg
        self.num_known = num_known
        self.embed = nn.Linear(raw_dim, cfg.d_model)
        self.embed_norm = nn.LayerNorm(cfg.d_model)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, cfg.d_model))
        self.context = ThreatContextEncoder(cfg)
        self.transformer = TransformerBranch(cfg) if cfg.use_transformer else None
        self.lstm = BiLSTMBranch(cfg) if cfg.use_lstm else None
        self.gate = VectorAdaptiveGate(cfg)

        self.mtm_head = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_model), nn.GELU(),
                                      nn.Linear(cfg.d_model, raw_dim))
        self.proj_head = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_model), nn.GELU(),
                                       nn.Linear(cfg.d_model, cfg.proj_dim))
        self.svdd = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_model, bias=False),
                                  nn.LeakyReLU(0.1),
                                  nn.Linear(cfg.d_model, cfg.svdd_dim, bias=False))
        self.refine = FeatureRefinement(cfg.d_model, cfg.dropout)
        self.classifier = nn.Linear(cfg.d_model, num_known)
        self.register_buffer("centers", torch.zeros(num_known, cfg.svdd_dim))
        self.register_buffer("temperature", torch.ones(1))

    def embed_seq(self, x, mtm_mask=None):
        e = self.embed_norm(self.embed(x))
        if mtm_mask is not None:
            m = mtm_mask.unsqueeze(-1).float()
            e = e * (1 - m) + self.mask_token * m
        return e

    def backbone(self, e, phi):
        r = e + phi.unsqueeze(1) if self.cfg.use_phi else e
        attns = None
        if self.transformer is not None:
            ht_seq, attns = self.transformer(r)
        else:
            ht_seq = torch.zeros_like(r)
        hl_seq = self.lstm(r) if self.lstm is not None else torch.zeros_like(r)
        ht, hl = ht_seq.mean(1), hl_seq.mean(1)
        a = self.gate(ht, hl, phi)
        z = a * ht + (1 - a) * hl
        f_seq = a.unsqueeze(1) * ht_seq + (1 - a.unsqueeze(1)) * hl_seq
        return dict(z=z, f_seq=f_seq, ht=ht, hl=hl, alpha=a, attns=attns)

    def encode(self, x, port, proto, mtm_mask=None):
        phi, c_pos, ms = self.context(port, proto)
        e = self.embed_seq(x, mtm_mask)
        out = self.backbone(e, phi)
        out["phi"] = phi
        out["mask_score"] = ms
        return out

    def logits(self, z):
        return self.classifier(self.refine(z))

    def svdd_dist(self, z):
        g = self.svdd(z)
        d = torch.cdist(g, self.centers) ** 2
        return d, g


def build_mask(scores, ratio, guided_frac):
    N, T = scores.shape
    n_mask = max(1, int(round(ratio * T)))
    n_guided = int(round(guided_frac * n_mask))
    mask = torch.zeros(N, T, dtype=torch.bool, device=scores.device)
    for b in range(N):
        order = torch.argsort(scores[b], descending=True)
        guided = order[:n_guided]
        rest = order[n_guided:]
        rand = rest[torch.randperm(rest.numel(), device=scores.device)][: n_mask - n_guided]
        idx = torch.cat([guided, rand])
        mask[b, idx] = True
    return mask


def nt_xent(z1, z2, temp):
    N = z1.size(0)
    z = F.normalize(torch.cat([z1, z2], dim=0), dim=1)
    sim = z @ z.t() / temp
    sim.fill_diagonal_(-1e9)
    targets = torch.cat([torch.arange(N, 2 * N), torch.arange(0, N)]).to(z.device)
    return F.cross_entropy(sim, targets)


def focal_loss(logits, target, gamma, weight=None):
    logp = F.log_softmax(logits, dim=1)
    p = logp.exp()
    ce = F.nll_loss(logp, target, weight=weight, reduction="none")
    pt = p.gather(1, target.unsqueeze(1)).squeeze(1)
    return ((1 - pt) ** gamma * ce).mean()


def loader(tensors, bs, shuffle):
    return DataLoader(TensorDataset(*tensors), batch_size=bs, shuffle=shuffle, drop_last=False)


def pretrain(model, train, cfg):
    if not cfg.use_ssl:
        return
    model.train()
    Xtr, Ptr, Rtr, _ = train
    dl = loader((Xtr, Ptr, Rtr), cfg.batch_size, True)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    for ep in range(cfg.pretrain_epochs):
        gf = min(1.0, ep / max(1, cfg.curriculum_warmup))
        tot = 0.0
        for x, p, r in dl:
            x, p, r = x.to(DEVICE), p.to(DEVICE), r.to(DEVICE)
            with torch.no_grad():
                _, _, ms = model.context(p, r)
            mask = build_mask(ms, cfg.mask_ratio, gf)
            out = model.encode(x, p, r, mtm_mask=mask)
            recon = model.mtm_head(out["f_seq"])
            m = mask.unsqueeze(-1).float()
            per = F.smooth_l1_loss(recon, x, reduction="none")
            l_mtm = (per * m).sum() / (m.sum().clamp_min(1.0) * x.size(-1))

            (x1, p1, r1), (x2, p2, r2) = augment(x, p, r)
            o1 = model.encode(x1, p1, r1)
            o2 = model.encode(x2, p2, r2)
            z1 = model.proj_head(o1["z"])
            z2 = model.proj_head(o2["z"])
            l_con = nt_xent(z1, z2, cfg.ntxent_temp)

            loss = cfg.lambda_mtm * l_mtm + cfg.lambda_contrast * l_con
            opt.zero_grad()
            loss.backward()
            opt.step()
            tot += loss.item() * x.size(0)
        print(f"[pretrain] epoch {ep + 1}/{cfg.pretrain_epochs} loss {tot / len(dl.dataset):.4f}")


@torch.no_grad()
def init_centers(model, train, label_map, cfg):
    model.eval()
    Xtr, Ptr, Rtr, Ytr = train
    dl = loader((Xtr, Ptr, Rtr, Ytr), cfg.batch_size, False)
    K = model.num_known
    sums = torch.zeros(K, cfg.svdd_dim, device=DEVICE)
    cnts = torch.zeros(K, device=DEVICE)
    for x, p, r, y in dl:
        x, p, r = x.to(DEVICE), p.to(DEVICE), r.to(DEVICE)
        ymap = torch.tensor([label_map[int(v)] for v in y], device=DEVICE)
        g = model.svdd(model.encode(x, p, r)["z"])
        for k in range(K):
            sel = ymap == k
            if sel.any():
                sums[k] += g[sel].sum(0)
                cnts[k] += sel.sum()
    centers = sums / cnts.clamp_min(1.0).unsqueeze(1)
    eps = 0.1
    centers[centers.abs() < eps] = eps
    model.centers.copy_(centers)


def train_supervised(model, train, label_map, cfg, ewc=None, replay=None):
    model.train()
    Xtr, Ptr, Rtr, Ytr = train
    ymap_all = torch.tensor([label_map[int(v)] for v in Ytr])
    counts = torch.bincount(ymap_all, minlength=model.num_known).float()
    weight = (counts.sum() / counts.clamp_min(1.0))
    weight = (weight / weight.sum() * model.num_known).to(DEVICE)
    dl = loader((Xtr, Ptr, Rtr, Ytr), cfg.batch_size, True)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    for ep in range(cfg.supervised_epochs):
        tot = 0.0
        for x, p, r, y in dl:
            x, p, r = x.to(DEVICE), p.to(DEVICE), r.to(DEVICE)
            ym = torch.tensor([label_map[int(v)] for v in y], device=DEVICE)
            if replay is not None and len(replay) > 0:
                rx, rp, rr, ry = replay.sample(cfg.replay_per_batch)
                x = torch.cat([x, rx.to(DEVICE)], 0)
                p = torch.cat([p, rp.to(DEVICE)], 0)
                r = torch.cat([r, rr.to(DEVICE)], 0)
                ym = torch.cat([ym, ry.to(DEVICE)], 0)
            out = model.encode(x, p, r)
            logits = model.logits(out["z"])
            l_cls = focal_loss(logits, ym, cfg.focal_gamma, weight)
            g = model.svdd(out["z"])
            c = model.centers[ym]
            l_svdd = ((g - c) ** 2).sum(1).mean()
            loss = l_cls + cfg.lambda_svdd * l_svdd
            if ewc is not None:
                loss = loss + cfg.ewc_lambda * ewc.penalty(model)
            opt.zero_grad()
            loss.backward()
            opt.step()
            tot += loss.item() * x.size(0)
        print(f"[supervised] epoch {ep + 1}/{cfg.supervised_epochs} loss {tot / len(dl.dataset):.4f}")


@torch.no_grad()
def gather_logits(model, tensors, label_map, cfg):
    model.eval()
    X, P, R, Y = tensors
    dl = loader((X, P, R, Y), cfg.batch_size, False)
    logits_all, dist_all, y_all, ymap_all = [], [], [], []
    for x, p, r, y in dl:
        x, p, r = x.to(DEVICE), p.to(DEVICE), r.to(DEVICE)
        out = model.encode(x, p, r)
        logits_all.append(model.logits(out["z"]).cpu())
        d, _ = model.svdd_dist(out["z"])
        dist_all.append(d.min(1).values.cpu())
        y_all.append(y)
        ymap_all.append(torch.tensor([label_map.get(int(v), -1) for v in y]))
    return (torch.cat(logits_all), torch.cat(dist_all),
            torch.cat(y_all), torch.cat(ymap_all))


def calibrate_temperature(model, val, label_map, cfg):
    logits, _, _, ymap = gather_logits(model, val, label_map, cfg)
    keep = ymap >= 0
    logits, ymap = logits[keep], ymap[keep]
    T = torch.nn.Parameter(torch.ones(1))
    opt = torch.optim.LBFGS([T], lr=0.1, max_iter=60)

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(logits / T.clamp_min(0.05), ymap)
        loss.backward()
        return loss

    opt.step(closure)
    model.temperature.copy_(T.detach().clamp_min(0.05).to(DEVICE))
    return float(model.temperature.item())


def compute_threshold(model, val, label_map, cfg):
    _, dist, _, ymap = gather_logits(model, val, label_map, cfg)
    known = dist[ymap >= 0].numpy()
    return float(np.quantile(known, cfg.svdd_quantile))


def expected_calibration_error(probs, labels, n_bins=15):
    conf, pred = probs.max(1)
    acc = (pred == labels).float()
    bins = torch.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i + 1])
        if m.any():
            ece += (m.float().mean() * (acc[m].mean() - conf[m].mean()).abs()).item()
    return ece


def evaluate(model, test, label_map, tau, cfg, known_classes):
    logits, dist, y_raw, ymap = gather_logits(model, test, label_map, cfg)
    probs = F.softmax(logits / model.temperature.cpu(), dim=1)
    is_unknown_true = (ymap < 0).long()
    is_unknown_pred = (dist > tau).long()

    osr_auc = float("nan")
    if is_unknown_true.sum() > 0 and is_unknown_true.sum() < len(is_unknown_true):
        osr_auc = roc_auc_score(is_unknown_true.numpy(), dist.numpy())
    osr_acc = accuracy_score(is_unknown_true.numpy(), is_unknown_pred.numpy())

    known_mask = (ymap >= 0) & (dist <= tau)
    res = {}
    if known_mask.any():
        pred = probs[known_mask].argmax(1)
        true = ymap[known_mask]
        res["known_acc"] = accuracy_score(true.numpy(), pred.numpy())
        res["known_f1_macro"] = f1_score(true.numpy(), pred.numpy(), average="macro", zero_division=0)
        res["ece"] = expected_calibration_error(probs[known_mask], ymap[known_mask])
    res["osr_auroc"] = osr_auc
    res["osr_acc"] = osr_acc
    res["unknown_recall"] = recall_score(is_unknown_true.numpy(), is_unknown_pred.numpy(), zero_division=0)
    res["known_count"] = int((ymap >= 0).sum())
    res["unknown_count"] = int((ymap < 0).sum())
    return res


@torch.no_grad()
def attention_rollout(model, x, p, r):
    model.eval()
    out = model.encode(x, p, r)
    attns = out["attns"]
    if attns is None:
        return None
    T = attns[0].size(-1)
    eye = torch.eye(T, device=attns[0].device).unsqueeze(0)
    roll = eye.repeat(attns[0].size(0), 1, 1)
    for a in attns:
        a = a + eye
        a = a / a.sum(-1, keepdim=True)
        roll = a @ roll
    importance = roll.mean(1)
    return importance


def integrated_gradients(model, x, p, r, target, steps=24):
    model.eval()
    baseline = torch.zeros_like(x)
    total = torch.zeros_like(x)
    with torch.backends.cudnn.flags(enabled=False):
        for s in range(1, steps + 1):
            xi = (baseline + (s / steps) * (x - baseline)).clone().requires_grad_(True)
            out = model.encode(xi, p, r)
            logit = model.logits(out["z"])[torch.arange(x.size(0)), target]
            grad = torch.autograd.grad(logit.sum(), xi)[0]
            total = total + grad
    attr = (x - baseline) * total / steps
    return attr.detach()


@torch.no_grad()
def fidelity_scores(model, x, p, r, target, ranking, fraction=0.5):
    model.eval()
    Fd = x.size(2)
    k = max(1, int(fraction * Fd))
    base = F.softmax(model.logits(model.encode(x, p, r)["z"]), 1)[torch.arange(x.size(0)), target]
    xd = x.clone()
    top = ranking.argsort(dim=1, descending=True)[:, :k]
    for b in range(x.size(0)):
        xd[b, :, top[b]] = 0.0
    deleted = F.softmax(model.logits(model.encode(xd, p, r)["z"]), 1)[torch.arange(x.size(0)), target]
    xi = torch.zeros_like(x)
    for b in range(x.size(0)):
        xi[b, :, top[b]] = x[b, :, top[b]]
    inserted = F.softmax(model.logits(model.encode(xi, p, r)["z"]), 1)[torch.arange(x.size(0)), target]
    return float((base - deleted).mean()), float(inserted.mean())


class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.data = []

    def __len__(self):
        return len(self.data)

    def add(self, x, p, r, y):
        for i in range(x.size(0)):
            item = (x[i].cpu(), p[i].cpu(), r[i].cpu(), int(y[i]))
            if len(self.data) < self.capacity:
                self.data.append(item)
            else:
                self.data[random.randint(0, self.capacity - 1)] = item

    def sample(self, n):
        n = min(n, len(self.data))
        batch = random.sample(self.data, n)
        x = torch.stack([b[0] for b in batch])
        p = torch.stack([b[1] for b in batch])
        r = torch.stack([b[2] for b in batch])
        y = torch.tensor([b[3] for b in batch], dtype=torch.long)
        return x, p, r, y


class EWC:
    def __init__(self, model, tensors, label_map, cfg):
        self.params = {n: pr.detach().clone() for n, pr in model.named_parameters() if pr.requires_grad}
        self.fisher = {n: torch.zeros_like(pr) for n, pr in self.params.items()}
        model.eval()
        X, P, R, Y = tensors
        dl = loader((X, P, R, Y), cfg.batch_size, True)
        n = 0
        with torch.backends.cudnn.flags(enabled=False):
            for x, p, r, y in dl:
                x, p, r = x.to(DEVICE), p.to(DEVICE), r.to(DEVICE)
                ym = torch.tensor([label_map[int(v)] for v in y], device=DEVICE)
                model.zero_grad()
                logits = model.logits(model.encode(x, p, r)["z"])
                loss = F.cross_entropy(logits, ym)
                loss.backward()
                for nm, pr in model.named_parameters():
                    if pr.grad is not None and nm in self.fisher:
                        self.fisher[nm] += pr.grad.detach() ** 2 * x.size(0)
                n += x.size(0)
        for nm in self.fisher:
            self.fisher[nm] /= max(1, n)

    def penalty(self, model):
        loss = 0.0
        for nm, pr in model.named_parameters():
            if nm in self.fisher and pr.shape == self.params[nm].shape:
                loss = loss + (self.fisher[nm] * (pr - self.params[nm]) ** 2).sum()
        return loss


def known_label_map(known_classes):
    return {c: i for i, c in enumerate(sorted(known_classes))}


def filter_known(tensors, known_classes):
    X, P, R, Y = tensors
    keep = torch.tensor([int(v) in known_classes for v in Y])
    return X[keep], P[keep], R[keep], Y[keep]


def run_pipeline(cfg, train, val, test, known_classes):
    raw_dim = train[0].size(2)
    label_map = known_label_map(known_classes)
    model = TARLNetV2(cfg, raw_dim, len(known_classes)).to(DEVICE)

    train_k = filter_known(train, known_classes)
    val_k = filter_known(val, known_classes)

    pretrain(model, train_k, cfg)
    init_centers(model, train_k, label_map, cfg)
    train_supervised(model, train_k, label_map, cfg)
    T = calibrate_temperature(model, val_k, label_map, cfg)
    tau = compute_threshold(model, val_k, label_map, cfg)
    print(f"[calibration] temperature {T:.3f}  threshold tau {tau:.3f}")
    res = evaluate(model, test, label_map, tau, cfg, known_classes)
    return model, label_map, tau, res


def run_loafo(cfg, train, val, test, held_family):
    held = {i for i, c in enumerate(CLASS_NAMES) if ATTACK_FAMILY[c] == held_family}
    known = {i for i in range(len(CLASS_NAMES)) if i not in held}
    print(f"\n=== LOAFO fold: holding out family '{held_family}' "
          f"({[CLASS_NAMES[i] for i in held]}) ===")
    model, label_map, tau, res = run_pipeline(cfg, train, val, test, known)
    print("LOAFO result:", {k: round(v, 4) if isinstance(v, float) else v for k, v in res.items()})
    return model, res


def run_ablation(cfg, train, val, test, held_family="recon"):
    held = {i for i, c in enumerate(CLASS_NAMES) if ATTACK_FAMILY[c] == held_family}
    known = {i for i in range(len(CLASS_NAMES)) if i not in held}
    variants = {
        "full": dict(),
        "no_transformer": dict(use_transformer=False),
        "no_lstm": dict(use_lstm=False),
        "no_phi": dict(use_phi=False),
        "no_ssl": dict(use_ssl=False),
        "scalar_gate": dict(vector_gate=False),
    }
    table = {}
    for name, overrides in variants.items():
        c = copy.deepcopy(cfg)
        for k, v in overrides.items():
            setattr(c, k, v)
        print(f"\n=== Ablation variant: {name} (open-set, holdout '{held_family}') ===")
        _, _, _, res = run_pipeline(c, train, val, test, known)
        table[name] = res
    print(f"\n=== Ablation summary (open-set, holdout '{held_family}') ===")
    for name, r in table.items():
        auroc = r.get("osr_auroc", float("nan"))
        auroc = f"{auroc:.4f}" if isinstance(auroc, float) and not math.isnan(auroc) else str(auroc)
        print(f"{name:16s} acc={r.get('known_acc', float('nan')):.4f} "
              f"f1={r.get('known_f1_macro', float('nan')):.4f} "
              f"osr_auroc={auroc} "
              f"unk_recall={r.get('unknown_recall', float('nan')):.4f}")
    return table


def demo_continual_learning(cfg, base_model, base_known, train, val, test):
    print("\n=== Continual learning demo (replay + EWC) ===")
    label_map = known_label_map(base_known)
    base_train = filter_known(train, base_known)
    ewc = EWC(base_model, base_train, label_map, cfg)
    replay = ReplayBuffer(cfg.replay_capacity)
    Xb, Pb, Rb, Yb = base_train
    yb_mapped = torch.tensor([label_map[int(v)] for v in Yb])
    replay.add(Xb[:cfg.replay_capacity], Pb[:cfg.replay_capacity],
               Rb[:cfg.replay_capacity], yb_mapped[:cfg.replay_capacity])

    missing = sorted(set(range(len(CLASS_NAMES))) - set(base_known))
    new_class = missing[0] if missing else None
    if new_class is None:
        print("no further class available for the demo")
        return base_model
    new_train = filter_known(train, {new_class})
    if new_train[0].shape[0] < 4:
        print(f"emerging class {CLASS_NAMES[new_class]} has too few samples; skipping")
        return base_model
    new_known = set(base_known) | {new_class}
    new_map = known_label_map(new_known)

    old_cls = base_model.classifier
    new_cls = nn.Linear(cfg.d_model, len(new_known)).to(DEVICE)
    with torch.no_grad():
        for c in base_known:
            new_cls.weight[new_map[c]] = old_cls.weight[label_map[c]]
            new_cls.bias[new_map[c]] = old_cls.bias[label_map[c]]
    base_model.classifier = new_cls
    base_model.num_known = len(new_known)
    new_centers = torch.zeros(len(new_known), cfg.svdd_dim, device=DEVICE)
    for c in base_known:
        new_centers[new_map[c]] = base_model.centers[label_map[c]]
    base_model.register_buffer("centers", new_centers)

    new_train = filter_known(train, {new_class})
    init_for_new = filter_known(train, new_known)
    init_centers(base_model, init_for_new, new_map, cfg)

    c = copy.deepcopy(cfg)
    c.supervised_epochs = max(4, cfg.supervised_epochs // 2)
    train_supervised(base_model, new_train, new_map, c, ewc=ewc, replay=replay)

    tau = compute_threshold(base_model, filter_known(val, new_known), new_map, cfg)
    res = evaluate(base_model, test, new_map, tau, cfg, new_known)
    print("after continual update:",
          {k: round(v, 4) if isinstance(v, float) else v for k, v in res.items()})
    return base_model


def demo_xai(model, test, label_map, cfg):
    print("\n=== XAI demo on one batch ===")
    X, P, R, Y = test
    keep = torch.tensor([int(v) in label_map for v in Y])
    X, P, R, Y = X[keep][:8], P[keep][:8], R[keep][:8], Y[keep][:8]
    x, p, r = X.to(DEVICE), P.to(DEVICE), R.to(DEVICE)
    target = torch.tensor([label_map[int(v)] for v in Y], device=DEVICE)
    roll = attention_rollout(model, x, p, r)
    if roll is not None:
        print("attention rollout (per-timestep importance, sample 0):",
              np.round(roll[0].cpu().numpy(), 3))
    attr = integrated_gradients(model, x, p, r, target)
    feat_rank = attr.abs().mean(1)
    top = feat_rank[0].argsort(descending=True)[:5].cpu().numpy()
    print("top critical features (sample 0):", top.tolist())
    deletion, insertion = fidelity_scores(model, x, p, r, target, feat_rank)
    print(f"fidelity  deletion(drop)={deletion:.4f}  insertion(retain)={insertion:.4f}")


def main():
    cfg = Config()
    print("device:", DEVICE)
    if cfg.dataset == "unsw":
        train, val, test, feat_idx, scaler = make_splits_unsw(cfg)
        held = "reconnaissance"
    else:
        set_taxonomy(["Normal", "DoS", "DDoS", "Probe", "Botnet",
                      "BruteForce", "WebAttack", "Infiltration"],
                     {"Normal": "benign", "DoS": "volumetric", "DDoS": "volumetric",
                      "Probe": "recon", "Botnet": "c2", "BruteForce": "credential",
                      "WebAttack": "webapp", "Infiltration": "intrusion"})
        train, val, test, feat_idx, scaler = make_splits(cfg)
        held = "recon"

    print(f"dataset={cfg.dataset}  classes={CLASS_NAMES}")
    print(f"sequences  train={tuple(train[0].shape)}  val={tuple(val[0].shape)}  test={tuple(test[0].shape)}")
    print(f"selected features ({len(feat_idx)}):", feat_idx.tolist())
    print("train label distribution:", torch.bincount(train[3], minlength=len(CLASS_NAMES)).tolist())

    known = set(range(len(CLASS_NAMES)))
    model, label_map, tau, res = run_pipeline(cfg, train, val, test, known)
    print("\n=== Closed-set baseline (all classes known) ===")
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in res.items()})

    demo_xai(model, test, label_map, cfg)
    run_loafo(cfg, train, val, test, held_family=held)

    base_known = set(range(len(CLASS_NAMES)))
    if cfg.dataset == "unsw" and "Fuzzers" in CLASS_NAMES:
        base_known = base_known - {CLASS_NAMES.index("Fuzzers")}
    else:
        base_known = set(range(len(CLASS_NAMES) - 1))
    base_model, base_map, base_tau, base_res = run_pipeline(cfg, train, val, test, base_known)
    demo_continual_learning(cfg, base_model, base_known, train, val, test)

    run_ablation(cfg, train, val, test, held_family=held)


In [2]:



















import time
try:
    from scipy.stats import weibull_min
    _HAS_WEIBULL = True
except Exception:
    _HAS_WEIBULL = False





@dataclass
class ConfigX(Config):

    seeds: tuple = (1337, 2024, 7)
    loafo_min_unknown: int = 20
    predict_horizon: int = 1
    openmax_tail: int = 20
    openmax_alpha: int = 10
    energy_T: float = 1.0
    osr_baseline_subset: int = 5000

    bench_batch: int = 256
    bench_iters: int = 20
    sweep_points: int = 41


def reseed(seed):
    global SEED
    SEED = int(seed)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)






CONTRIBUTIONS = """
CORE CONTRIBUTION (single, foregrounded):
    Threat-Context-Conditioned Adaptive Fusion.
    A per-dimension (vector) gate that fuses the Transformer and BiLSTM branches,
    *conditioned on a learnable Threat-Context vector phi_c*, where the SAME phi_c
    also drives the curriculum masking strategy of the self-supervised pretext task.
    One mechanism, used at two points (fusion gate + mask guidance) — this is the
    novel contribution around which the whole architecture is organised.

SUPPORTING COMPONENTS (explicitly NOT claimed as standalone novelty):
    - Self-supervised pretraining (MTM curriculum + semantic-preserving contrastive)
    - Class-conditional Deep-SVDD open-world gate with anti-collapse + quantile threshold
    - Continual-learning loop (Replay + EWC), analyst-in-the-loop
    - Operational Trust Layer  (renamed from "Explainability"):
        Attention-Rollout (online) + Integrated-Gradients (offline) + Temperature
        calibration, validated by deletion/insertion fidelity and ECE. This is a
        *trust/validation* capability that hardens the decisions for SOC operators,
        NOT a novelty claim.
"""

THREAT_MODEL = """
THREAT MODEL (scope statement — defines what the system defends against; no new
adversarial experiments are claimed here):

  Assets ............ network flows of (host, service) pairs; the IDS decision pipeline.
  Defender .......... a SOC operator running TARL-Net v2 on flow-level features only.
  Attacker goal ..... evade detection and/or operate via an attack family never seen
                      in training (zero-day), violating Integrity / Availability.
  Attacker knowledge: IN-SCOPE  = black-box w.r.t. the model (no access to weights,
                      gradients, thresholds); may craft novel families and obfuscate.
                      OUT-OF-SCOPE = white-box adaptive adversary with full model
                      access and gradient-based evasion (future work).
  Poisoning ......... mitigated by design: model updates occur ONLY through the
                      analyst-in-the-loop continual-learning path (no automatic
                      pseudo-labels), so a single attacker cannot self-label data
                      into the replay buffer. Targeted training-set poisoning of the
                      public dataset is OUT-OF-SCOPE.
  Evasion ........... volumetric/pattern obfuscation is partially covered by the
                      semantic-preserving augmentations; gradient-based adversarial
                      perturbation robustness is OUT-OF-SCOPE and flagged as future work.
  Label leakage ..... signature-equivalent features are removed from phi_c by the
                      leakage audit so the context encoder cannot shortcut the label.
"""

WORDING_FIX_NOTE = (
    'THESIS WORDING FIX (§4-4): replace the word "واگرا" (divergent) with '
    '"به‌صورت یکنواخت و کاهشی همگرا می‌شود" — the supervised loss DECREASES '
    '(0.78 -> 0.22), so "convergent/decreasing" is the correct term.'
)


def print_documentation():
    print("\n" + "=" * 78)
    print(CONTRIBUTIONS)
    print(THREAT_MODEL)
    print(WORDING_FIX_NOTE)
    print("=" * 78 + "\n")








def build_window_indices_pred(data, cfg, horizon):
    y, host, service, ts = data["y"], data["host"], data["service"], data["ts"]
    groups = {}
    for i in range(len(y)):
        groups.setdefault((int(host[i]), int(service[i])), []).append(i)
    windows = []
    for _, idxs in groups.items():
        idxs.sort(key=lambda j: ts[j])
        last = len(idxs) - cfg.seq_len - horizon
        for s in range(0, last + 1, cfg.seq_stride):
            win = idxs[s:s + cfg.seq_len]
            label_pos = idxs[s + cfg.seq_len - 1 + horizon]
            windows.append((np.asarray(win, dtype=np.int64), int(y[label_pos])))
    return windows


def make_splits_unsw_pred(cfg):
    data = load_unsw_nb15(cfg)
    windows = build_window_indices_pred(data, cfg, max(0, cfg.predict_horizon))
    tr, va, te = stratified_split_windows(windows, cfg)
    tr = balance_train_indices(tr, windows, cfg)
    train_flows = np.unique(np.concatenate([windows[i][0] for i in tr]))
    Xtr_flat, lo, hi, med = impute_and_clip(data["X"][train_flows])
    ytr_flat = data["y"][train_flows]
    if len(Xtr_flat) > 60000:
        sub = np.random.RandomState(SEED).choice(len(Xtr_flat), 60000, replace=False)
        feat_idx = select_features(Xtr_flat[sub], ytr_flat[sub], cfg.mi_keep, cfg.corr_thresh)
    else:
        feat_idx = select_features(Xtr_flat, ytr_flat, cfg.mi_keep, cfg.corr_thresh)
    scaler = RobustScaler().fit(Xtr_flat[:, feat_idx])
    data["X"], _, _, _ = impute_and_clip(data["X"], lo, hi, med)
    train = assemble_windows(data, windows, tr, feat_idx, scaler)
    val = assemble_windows(data, windows, va, feat_idx, scaler)
    test = assemble_windows(data, windows, te, feat_idx, scaler)
    return train, val, test, feat_idx, scaler


def build_sequences_pred(data, feat_idx, scaler, cfg, host_set, horizon):
    X = scaler.transform(data["X"][:, feat_idx])
    y, host, service = data["y"], data["host"], data["service"]
    port, proto, ts = data["port"], data["proto"], data["ts"]
    seqs, sports, sprotos, slabels = [], [], [], []
    keys = {}
    for i in range(len(y)):
        if host[i] not in host_set:
            continue
        keys.setdefault((host[i], service[i]), []).append(i)
    for _, idxs in keys.items():
        idxs = sorted(idxs, key=lambda j: ts[j])
        last = len(idxs) - cfg.seq_len - horizon
        for s in range(0, last + 1, cfg.seq_stride):
            win = idxs[s:s + cfg.seq_len]
            label_pos = idxs[s + cfg.seq_len - 1 + horizon]
            seqs.append(X[win]); sports.append(port[win])
            sprotos.append(proto[win]); slabels.append(y[label_pos])
    return (torch.tensor(np.asarray(seqs), dtype=torch.float32),
            torch.tensor(np.asarray(sports), dtype=torch.long),
            torch.tensor(np.asarray(sprotos), dtype=torch.long),
            torch.tensor(np.asarray(slabels), dtype=torch.long))


def make_splits_pred(cfg):
    data = generate_flows(cfg)
    rng = np.random.RandomState(SEED)
    hosts = np.arange(cfg.n_hosts); rng.shuffle(hosts)
    n_tr = int(0.6 * cfg.n_hosts); n_va = int(0.2 * cfg.n_hosts)
    tr_h = set(hosts[:n_tr].tolist())
    va_h = set(hosts[n_tr:n_tr + n_va].tolist())
    te_h = set(hosts[n_tr + n_va:].tolist())
    Xtr_flat = data["X"][np.isin(data["host"], list(tr_h))]
    ytr_flat = data["y"][np.isin(data["host"], list(tr_h))]
    Xtr_flat, lo, hi, med = impute_and_clip(Xtr_flat)
    feat_idx = select_features(Xtr_flat, ytr_flat, cfg.mi_keep, cfg.corr_thresh)
    scaler = RobustScaler().fit(Xtr_flat[:, feat_idx])
    data["X"], _, _, _ = impute_and_clip(data["X"], lo, hi, med)
    h = max(0, cfg.predict_horizon)
    train = build_sequences_pred(data, feat_idx, scaler, cfg, tr_h, h)
    val = build_sequences_pred(data, feat_idx, scaler, cfg, va_h, h)
    test = build_sequences_pred(data, feat_idx, scaler, cfg, te_h, h)
    return train, val, test, feat_idx, scaler


@torch.no_grad()
def early_warning_scores(model, tensors, label_map, cfg):
    logits, dist, y_raw, ymap = gather_logits(model, tensors, label_map, cfg)
    probs = F.softmax(logits / model.temperature.cpu(), dim=1)
    normal_idx = label_map.get(NORMAL_LABEL, None)
    if normal_idx is None:
        ews = 1.0 - probs.max(1).values
    else:
        ews = 1.0 - probs[:, normal_idx]
    return ews, ymap, normal_idx


def run_prediction_demo(cfg, pred_builder):
    print(f"\n=== Cyber-Attack PREDICTION (horizon H={max(1, cfg.predict_horizon)}) ===")
    cfgp = copy.deepcopy(cfg)
    if cfgp.predict_horizon <= 0:
        cfgp.predict_horizon = 1
    train, val, test, _, _ = pred_builder(cfgp)
    known = set(range(len(CLASS_NAMES)))
    model, label_map, tau, res = run_pipeline(cfgp, train, val, test, known)
    ews, ymap, normal_idx = early_warning_scores(model, test, label_map, cfgp)
    keep = ymap >= 0
    y_attack = (ymap[keep] != normal_idx).numpy().astype(int) if normal_idx is not None else None
    ews_auroc = float("nan")
    if y_attack is not None and 0 < y_attack.sum() < len(y_attack):
        ews_auroc = roc_auc_score(y_attack, ews[keep].numpy())
    print(f"predictive next-step classification: acc={res.get('known_acc', float('nan')):.4f} "
          f"macro-F1={res.get('known_f1_macro', float('nan')):.4f}")
    print(f"early-warning score AUROC (attack-vs-normal, H={cfgp.predict_horizon}): {ews_auroc:.4f}")
    return model, res, ews_auroc





def energy_unknown_score(logits, T=1.0):
    return (-T * torch.logsumexp(logits / T, dim=1))


def msp_unknown_score(logits):
    return 1.0 - F.softmax(logits, dim=1).max(1).values


class OpenMax:
    def __init__(self, tail=20, alpha=10):
        self.tail = tail
        self.alpha = alpha
        self.mav = {}
        self.wb = {}

    def fit(self, logits, ymap, K):
        logits = np.asarray(logits, dtype=np.float64)
        ymap = np.asarray(ymap)
        pred = logits.argmax(1)
        for k in range(K):
            sel = (ymap == k) & (pred == k)
            if sel.sum() < 5:
                sel = (ymap == k)
            if sel.sum() < 5:
                self.mav[k] = logits[ymap == k].mean(0) if (ymap == k).any() else np.zeros(logits.shape[1])
                self.wb[k] = (1.0, 1.0)
                continue
            mav = logits[sel].mean(0)
            d = np.linalg.norm(logits[sel] - mav, axis=1)
            tail = np.sort(d)[-min(self.tail, len(d)):]
            self.mav[k] = mav
            if _HAS_WEIBULL and len(tail) >= 3 and tail.max() > tail.min():
                try:
                    c, loc, scale = weibull_min.fit(tail, floc=0)
                    self.wb[k] = (float(c), float(max(scale, 1e-6)))
                except Exception:
                    self.wb[k] = (1.0, float(max(tail.mean(), 1e-6)))
            else:
                self.wb[k] = (1.0, float(max(tail.mean(), 1e-6)))
        return self

    def _wscore(self, k, dist):
        c, scale = self.wb.get(k, (1.0, 1.0))
        if _HAS_WEIBULL:
            try:
                return float(weibull_min.cdf(dist, c, loc=0, scale=scale))
            except Exception:
                pass
        return float(1.0 - math.exp(-(dist / max(scale, 1e-6)) ** c))

    def unknown_score_batch(self, logits):
        logits = np.asarray(logits, dtype=np.float64)
        K = logits.shape[1]
        a = min(self.alpha, K)
        out = np.zeros(len(logits))
        for i, v in enumerate(logits):
            ranked = np.argsort(v)[::-1][:a]
            w = np.ones(K)
            for rank, j in enumerate(ranked):
                d = float(np.linalg.norm(v - self.mav.get(j, np.zeros(K))))
                ws = self._wscore(j, d)
                w[j] = 1.0 - (a - rank) / a * ws
            v_hat = v * w
            unknown_logit = float((v * (1.0 - w)).sum())
            ex = np.exp(np.concatenate([v_hat, [unknown_logit]]) - np.max(np.append(v_hat, unknown_logit)))
            out[i] = ex[-1] / ex.sum()
        return out


def compare_osr_baselines(model, train_known, test, label_map, cfg):
    print("\n=== Open-set baselines (unknown-detection AUROC, same fold) ===")
    logits_tr, dist_tr, _, ymap_tr = gather_logits(model, train_known, label_map, cfg)
    logits_te, dist_te, _, ymap_te = gather_logits(model, test, label_map, cfg)
    is_unk = (ymap_te < 0).numpy().astype(int)
    if is_unk.sum() == 0 or is_unk.sum() == len(is_unk):
        print("  (no unknown samples in this split — run under a LOAFO fold)")
        return {}

    n = len(is_unk)
    sub = np.arange(n)
    if n > cfg.osr_baseline_subset:
        sub = np.random.RandomState(SEED).choice(n, cfg.osr_baseline_subset, replace=False)
    yb = is_unk[sub]

    scores = {
        "SVDD (ours)": dist_te.numpy()[sub],
        "Energy-OOD": energy_unknown_score(logits_te, cfg.energy_T).numpy()[sub],
        "MSP": msp_unknown_score(logits_te).numpy()[sub],
    }
    om = OpenMax(cfg.openmax_tail, cfg.openmax_alpha).fit(
        logits_tr.numpy(), ymap_tr.numpy(), model.num_known)
    scores["OpenMax"] = om.unknown_score_batch(logits_te.numpy()[sub])

    out = {}
    for name, s in scores.items():
        if 0 < yb.sum() < len(yb):
            out[name] = float(roc_auc_score(yb, s))
            print(f"  {name:14s} AUROC = {out[name]:.4f}")
    return out





def estimate_flops_per_window(cfg, raw_dim):
    T, d = cfg.seq_len, cfg.d_model
    ff = cfg.ff_dim
    flops = 0
    flops += 2 * T * raw_dim * d
    flops += 2 * T * (2 * cfg.ctx_emb_dim) * d
    if cfg.use_transformer:
        per_layer = (3 * 2 * T * d * d
                     + 2 * T * T * d
                     + 2 * T * T * d
                     + 2 * T * d * d
                     + 2 * 2 * T * d * ff)
        flops += cfg.n_transformer_layers * per_layer
    if cfg.use_lstm:
        flops += cfg.n_lstm_layers * 2 * T * 8 * cfg.lstm_hidden * (d + cfg.lstm_hidden)
        flops += 2 * T * (2 * cfg.lstm_hidden) * d
    flops += 2 * (3 * d) * (d if cfg.vector_gate else 1)
    flops += 2 * d * d + 2 * d * cfg.num_known if hasattr(cfg, "num_known") else 2 * d * d
    flops += 2 * d * d + 2 * d * cfg.svdd_dim
    return int(flops)


@torch.no_grad()
def operational_metrics(model, cfg, raw_dim):
    print("\n=== Operational metrics ===")
    model.eval()
    n_params = sum(p.numel() for p in model.parameters())
    B, T = cfg.bench_batch, cfg.seq_len
    x = torch.randn(B, T, raw_dim, device=DEVICE)
    p = torch.randint(0, cfg.n_port_buckets, (B, T), device=DEVICE)
    r = torch.randint(0, cfg.n_protocols, (B, T), device=DEVICE)
    for _ in range(3):
        out = model.encode(x, p, r); model.logits(out["z"]); model.svdd_dist(out["z"])
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    for _ in range(cfg.bench_iters):
        out = model.encode(x, p, r); model.logits(out["z"]); model.svdd_dist(out["z"])
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) / cfg.bench_iters
    latency_ms = dt / B * 1000.0
    throughput = B / dt
    if DEVICE.type == "cuda":
        mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    else:
        mem_mb = n_params * 4 / (1024 ** 2)
    flops = estimate_flops_per_window(cfg, raw_dim)
    res = dict(params=int(n_params),
               approx_flops_per_window=int(flops),
               latency_ms_per_window=round(latency_ms, 4),
               throughput_windows_per_s=round(throughput, 1),
               peak_mem_mb=round(mem_mb, 2),
               device=DEVICE.type)
    print(f"  parameters ............ {res['params']:,}")
    print(f"  approx FLOPs / window . {res['approx_flops_per_window']:,}")
    print(f"  latency / window ...... {res['latency_ms_per_window']:.4f} ms")
    print(f"  throughput ............ {res['throughput_windows_per_s']:.1f} windows/s")
    print(f"  peak memory ........... {res['peak_mem_mb']:.2f} MB ({res['device']})")
    return res





@torch.no_grad()
def threshold_sweep(model, test, label_map, cfg, plot=False, save_path=None):
    print("\n=== Threshold sweep (false-alarm vs unknown-recall) ===")
    logits, dist, y_raw, ymap = gather_logits(model, test, label_map, cfg)
    d = dist.numpy()
    unk = (ymap < 0).numpy().astype(bool)
    if unk.sum() == 0 or (~unk).sum() == 0:
        print("  (need a LOAFO fold with both known and unknown samples)")
        return []
    known_d = d[~unk]
    qs = np.linspace(0.50, 0.999, cfg.sweep_points)
    taus = np.quantile(known_d, qs)
    rows = []
    for q, tau in zip(qs, taus):
        pred_unk = d > tau
        rec = float(pred_unk[unk].mean())
        fpr = float(pred_unk[~unk].mean())
        rows.append((float(q), float(tau), fpr, rec))
    print("  quantile   tau      FPR(false-alarm)  unknown-recall")
    for q, tau, fpr, rec in rows[::max(1, len(rows) // 8)]:
        print(f"   {q:5.3f}  {tau:7.4f}      {fpr:6.4f}           {rec:6.4f}")
    if plot:
        try:
            import matplotlib.pyplot as plt
            fpr = [x[2] for x in rows]; rec = [x[3] for x in rows]
            plt.figure(figsize=(5, 4))
            plt.plot(fpr, rec, marker="o", ms=3)
            plt.xlabel("False-alarm rate on known traffic (FPR)")
            plt.ylabel("Unknown / zero-day recall")
            plt.title("Open-world operating-point trade-off")
            plt.grid(True, alpha=0.3)
            if save_path:
                plt.savefig(save_path, dpi=130, bbox_inches="tight")
            plt.show()
        except Exception as e:
            print("  (plot skipped:", e, ")")
    return rows





@torch.no_grad()
def gate_error_propagation(model, test, label_map, tau, cfg):
    print("\n=== Gate error-propagation analysis ===")
    logits, dist, y_raw, ymap = gather_logits(model, test, label_map, cfg)
    probs = F.softmax(logits / model.temperature.cpu(), dim=1)
    is_unk = (ymap < 0)
    pred_unk = (dist > tau)
    pred_cls = probs.argmax(1)
    cls_correct = (pred_cls == ymap)

    gate_fpr = float(pred_unk[~is_unk].float().mean()) if (~is_unk).any() else float("nan")
    gate_fnr = float((~pred_unk[is_unk]).float().mean()) if is_unk.any() else float("nan")

    actual_correct = torch.where(is_unk, pred_unk, (~pred_unk) & cls_correct)
    e2e_acc = float(actual_correct.float().mean())
    oracle_correct = torch.where(is_unk, torch.ones_like(pred_unk), cls_correct)
    oracle_acc = float(oracle_correct.float().mean())

    res = dict(gate_fpr_known_to_unknown=round(gate_fpr, 4),
               gate_fnr_unknown_to_known=round(gate_fnr, 4),
               end_to_end_acc=round(e2e_acc, 4),
               oracle_gate_acc=round(oracle_acc, 4),
               error_attributable_to_gate=round(oracle_acc - e2e_acc, 4))
    for k, v in res.items():
        print(f"  {k:32s} = {v}")
    return res





def _count_unknown_test(test, held_classes):
    return int(sum(int(v) in held_classes for v in test[3]))


def run_loafo_all(cfg, train, val, test):
    families = sorted({ATTACK_FAMILY[c] for c in CLASS_NAMES if ATTACK_FAMILY[c] != "benign"})
    results = {}
    for fam in families:
        held = {i for i, c in enumerate(CLASS_NAMES) if ATTACK_FAMILY[c] == fam}
        n_unk = _count_unknown_test(test, held)
        if n_unk < cfg.loafo_min_unknown:
            print(f"[LOAFO] skip '{fam}' (only {n_unk} unknown test windows < {cfg.loafo_min_unknown})")
            continue
        known = {i for i in range(len(CLASS_NAMES)) if i not in held}
        _, _, _, res = run_pipeline(cfg, train, val, test, known)
        results[fam] = res
        print(f"[LOAFO] {fam:16s} auroc={res.get('osr_auroc', float('nan')):.4f}  "
              f"unk_recall={res.get('unknown_recall', float('nan')):.4f}  "
              f"known_F1={res.get('known_f1_macro', float('nan')):.4f}")
    return results


def _mean_std(values):
    vals = [v for v in values if isinstance(v, (int, float)) and not (isinstance(v, float) and math.isnan(v))]
    if not vals:
        return float("nan"), float("nan")
    return float(np.mean(vals)), float(np.std(vals))


def run_loafo_multiseed(cfg, builder, seeds):
    print("\n" + "#" * 78)
    print(f"# FULL LOAFO  x  MULTI-SEED   seeds={list(seeds)}")
    print("#" * 78)
    per_seed = {}
    for sd in seeds:
        print(f"\n----- seed {sd} -----")
        reseed(sd)
        train, val, test, _, _ = builder(cfg)
        per_seed[sd] = run_loafo_all(cfg, train, val, test)

    families = sorted({f for d in per_seed.values() for f in d})
    print("\n=== Per-family results across seeds (mean ± std) ===")
    print(f"{'family':16s} {'OSR-AUROC':>16s} {'unknown-recall':>18s} {'known-F1':>16s}")
    summary = {}
    for fam in families:
        auroc = [per_seed[s][fam]["osr_auroc"] for s in per_seed if fam in per_seed[s]]
        rec = [per_seed[s][fam]["unknown_recall"] for s in per_seed if fam in per_seed[s]]
        f1 = [per_seed[s][fam]["known_f1_macro"] for s in per_seed if fam in per_seed[s]]
        am, asd = _mean_std(auroc); rm, rsd = _mean_std(rec); fm, fsd = _mean_std(f1)
        summary[fam] = dict(auroc=(am, asd), unknown_recall=(rm, rsd), known_f1=(fm, fsd))
        print(f"{fam:16s} {am:7.4f} ± {asd:6.4f} {rm:8.4f} ± {rsd:6.4f} {fm:7.4f} ± {fsd:6.4f}")

    all_auroc = [r["osr_auroc"] for d in per_seed.values() for r in d.values()]
    all_rec = [r["unknown_recall"] for d in per_seed.values() for r in d.values()]
    all_f1 = [r["known_f1_macro"] for d in per_seed.values() for r in d.values()]
    gm = {"OSR-AUROC": _mean_std(all_auroc),
          "unknown-recall": _mean_std(all_rec),
          "known-macro-F1": _mean_std(all_f1)}
    print("\n=== Overall (all folds x all seeds) ===")
    for k, (m, s) in gm.items():
        print(f"  {k:16s} = {m:.4f} ± {s:.4f}")
    return per_seed, summary, gm





SAMPLE_LLM_REPORT = """\
[SAMPLE / ILLUSTRATIVE LLM OUTPUT — not produced by an evaluated model]
INCIDENT SUMMARY
  • Verdict ........ Probable ZERO-DAY (unknown family), severity HIGH.
  • Rationale ...... Representation lies far outside every known-class hypersphere
                     (anomaly score well above the 95th-percentile gate threshold).
  • Key indicators . Destination-port-bucket risk and connection-state features
                     dominate the attribution; temporal weight concentrates on the
                     final third of the window (burst-like behaviour).
ANALYST ACTIONS
  1. Isolate the (host, service) pair and capture full PCAP for the flagged window.
  2. Correlate the suspicious destination-port bucket with current threat intel.
  3. If confirmed malicious, label and submit via the analyst-in-the-loop queue so
     the continual-learning buffer can absorb the new family without forgetting.
"""


def build_forensic_prompt(phi_vec, anomaly_score, severity, top_feature_idx,
                          top_feature_vals, feat_global_idx=None):
    phi = np.asarray(phi_vec).ravel()
    phi_summary = dict(dim=int(phi.shape[0]),
                       l2_norm=round(float(np.linalg.norm(phi)), 4),
                       mean=round(float(phi.mean()), 4),
                       max_abs=round(float(np.abs(phi).max()), 4))
    feats = []
    for rank, (fi, fv) in enumerate(zip(top_feature_idx, top_feature_vals), 1):
        gid = int(feat_global_idx[fi]) if feat_global_idx is not None else int(fi)
        feats.append(f"#{rank} feature[selected_idx={int(fi)}, dataset_col={gid}] "
                     f"attribution={float(fv):+.4f}")
    prompt = f"""You are a SOC forensic-analysis assistant. A flow-based IDS (TARL-Net v2)
flagged a window as a possible ZERO-DAY attack via its class-conditional open-world gate.

[THREAT-CONTEXT ENCODER OUTPUT phi_c]
  {phi_summary}

[OPEN-WORLD GATE]
  anomaly_score (min distance to nearest known-class center) = {float(anomaly_score):.4f}
  severity = {severity}

[SUSPICIOUS-FEATURE PROFILE  (Integrated-Gradients attribution, top-k)]
  {chr(10) + '  '.join(feats)}

TASK:
  1. Give a one-line verdict (known-family / zero-day / benign-anomaly) with confidence.
  2. Explain, in <=4 bullets, which indicators drive the verdict.
  3. List concrete analyst next-steps and a containment recommendation.
  4. State what additional context you would request to raise confidence.
Return a concise, operator-ready incident note."""
    return prompt


@torch.no_grad()
def _phi_for_sample(model, x, p, r):
    out = model.encode(x, p, r)
    return out["phi"].cpu().numpy()


def demo_forensic_llm(model, test, label_map, cfg, feat_idx=None):
    print("\n=== LLM forensic use-case (ILLUSTRATIVE — prompt builder + sample report) ===")
    X, P, R, Y = test
    n = min(2048, X.size(0))
    x, p, r = X[:n].to(DEVICE), P[:n].to(DEVICE), R[:n].to(DEVICE)
    out = model.encode(x, p, r)
    dist, _ = model.svdd_dist(out["z"])
    dmin = dist.min(1).values
    j = int(dmin.argmax())
    xj, pj, rj = x[j:j + 1], p[j:j + 1], r[j:j + 1]
    logits = model.logits(model.encode(xj, pj, rj)["z"])
    target = logits.argmax(1)
    attr = integrated_gradients(model, xj, pj, rj, target)
    feat_rank = attr.abs().mean(1)[0]
    k = min(5, feat_rank.numel())
    top = feat_rank.argsort(descending=True)[:k].cpu().numpy()
    top_vals = feat_rank[top].cpu().numpy()
    phi = _phi_for_sample(model, xj, pj, rj)
    sev = "HIGH" if float(dmin[j]) > float(dmin.median()) else "MEDIUM"
    prompt = build_forensic_prompt(phi, float(dmin[j]), sev, top, top_vals,
                                   feat_global_idx=(feat_idx if feat_idx is not None else None))
    print(prompt)
    print("\n" + SAMPLE_LLM_REPORT)
    print("NOTE: wiring this prompt to a real LLM endpoint is left as deployment/future work;\n"
          "      no LLM output here is an evaluated contribution of the thesis.")
    return prompt





def main_full(cfg=None):
    cfg = cfg or ConfigX()
    print("device:", DEVICE)
    print_documentation()


    if cfg.dataset == "unsw":
        builder = make_splits_unsw
        pred_builder = make_splits_unsw_pred
        held = "reconnaissance"
    else:
        set_taxonomy(["Normal", "DoS", "DDoS", "Probe", "Botnet",
                      "BruteForce", "WebAttack", "Infiltration"],
                     {"Normal": "benign", "DoS": "volumetric", "DDoS": "volumetric",
                      "Probe": "recon", "Botnet": "c2", "BruteForce": "credential",
                      "WebAttack": "webapp", "Infiltration": "intrusion"})
        builder = make_splits
        pred_builder = make_splits_pred
        held = "recon"


    reseed(cfg.seeds[0])
    train, val, test, feat_idx, scaler = builder(cfg)
    print(f"\ndataset={cfg.dataset}  classes={CLASS_NAMES}")
    print(f"sequences  train={tuple(train[0].shape)}  val={tuple(val[0].shape)}  test={tuple(test[0].shape)}")
    print(f"selected features ({len(feat_idx)}):", feat_idx.tolist())
    print("train label distribution:", torch.bincount(train[3], minlength=len(CLASS_NAMES)).tolist())


    known = set(range(len(CLASS_NAMES)))
    model, label_map, tau, res = run_pipeline(cfg, train, val, test, known)
    print("\n=== Closed-set baseline (all classes known) ===")
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in res.items()})
    demo_xai(model, test, label_map, cfg)


    operational_metrics(model, cfg, train[0].size(2))


    held_set = {i for i, c in enumerate(CLASS_NAMES) if ATTACK_FAMILY[c] == held}
    known_l = {i for i in range(len(CLASS_NAMES)) if i not in held_set}
    mdl_l, lm_l, tau_l, res_l = run_pipeline(cfg, train, val, test, known_l)
    print(f"\n=== Reference LOAFO fold (holdout '{held}') ===")
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in res_l.items()})

    threshold_sweep(mdl_l, test, lm_l, cfg, plot=False)
    gate_error_propagation(mdl_l, test, lm_l, tau_l, cfg)
    compare_osr_baselines(mdl_l, filter_known(train, known_l), test, lm_l, cfg)
    demo_forensic_llm(mdl_l, test, lm_l, cfg, feat_idx)


    run_loafo_multiseed(cfg, builder, cfg.seeds)


    run_prediction_demo(cfg, pred_builder)


    base_known = set(range(len(CLASS_NAMES)))
    if cfg.dataset == "unsw" and "Fuzzers" in CLASS_NAMES:
        base_known = base_known - {CLASS_NAMES.index("Fuzzers")}
    else:
        base_known = set(range(len(CLASS_NAMES) - 1))
    base_model, base_map, base_tau, base_res = run_pipeline(cfg, train, val, test, base_known)
    demo_continual_learning(cfg, base_model, base_known, train, val, test)


    run_ablation(cfg, train, val, test, held_family=held)


In [3]:
















def run_loafo_all_with_baselines(cfg, train, val, test):
    families = sorted({ATTACK_FAMILY[c] for c in CLASS_NAMES if ATTACK_FAMILY[c] != "benign"})
    results, baselines = {}, {}
    for fam in families:
        held = {i for i, c in enumerate(CLASS_NAMES) if ATTACK_FAMILY[c] == fam}
        n_unk = _count_unknown_test(test, held)
        if n_unk < cfg.loafo_min_unknown:
            print(f"[LOAFO] skip '{fam}' (only {n_unk} unknown test windows < {cfg.loafo_min_unknown})")
            continue
        known = {i for i in range(len(CLASS_NAMES)) if i not in held}
        model, label_map, tau, res = run_pipeline(cfg, train, val, test, known)
        results[fam] = res
        try:
            baselines[fam] = compare_osr_baselines(model, filter_known(train, known),
                                                   test, label_map, cfg)
        except Exception as e:
            print(f"  (baseline comparison skipped for '{fam}': {e})")
            baselines[fam] = {}
        print(f"[LOAFO] {fam:16s} auroc={res.get('osr_auroc', float('nan')):.4f}  "
              f"unk_recall={res.get('unknown_recall', float('nan')):.4f}  "
              f"known_F1={res.get('known_f1_macro', float('nan')):.4f}")
    return results, baselines


def run_loafo_multiseed_v2(cfg, builder, seeds):
    print("\n" + "#" * 78)
    print(f"# FULL LOAFO  x  MULTI-SEED  (+ open-set baselines)   seeds={list(seeds)}")
    print("#" * 78)
    per_seed, per_seed_bl = {}, {}
    for sd in seeds:
        print(f"\n----- seed {sd} -----")
        reseed(sd)
        train, val, test, _, _ = builder(cfg)
        r, b = run_loafo_all_with_baselines(cfg, train, val, test)
        per_seed[sd] = r
        per_seed_bl[sd] = b


    families = sorted({f for d in per_seed.values() for f in d})
    print("\n=== Per-family results across seeds (mean ± std) ===")
    print(f"{'family':16s} {'OSR-AUROC':>16s} {'unknown-recall':>18s} {'known-F1':>16s}")
    summary = {}
    for fam in families:
        au = [per_seed[s][fam]["osr_auroc"] for s in per_seed if fam in per_seed[s]]
        rc = [per_seed[s][fam]["unknown_recall"] for s in per_seed if fam in per_seed[s]]
        f1 = [per_seed[s][fam]["known_f1_macro"] for s in per_seed if fam in per_seed[s]]
        am, asd = _mean_std(au); rm, rsd = _mean_std(rc); fm, fsd = _mean_std(f1)
        summary[fam] = dict(auroc=(am, asd), unknown_recall=(rm, rsd), known_f1=(fm, fsd))
        print(f"{fam:16s} {am:7.4f} ± {asd:6.4f} {rm:8.4f} ± {rsd:6.4f} {fm:7.4f} ± {fsd:6.4f}")

    all_au = [r["osr_auroc"] for d in per_seed.values() for r in d.values()]
    all_rc = [r["unknown_recall"] for d in per_seed.values() for r in d.values()]
    all_f1 = [r["known_f1_macro"] for d in per_seed.values() for r in d.values()]
    print("\n=== Overall (all folds x all seeds) ===")
    for name, vals in [("OSR-AUROC", all_au), ("unknown-recall", all_rc), ("known-macro-F1", all_f1)]:
        m, s = _mean_std(vals)
        print(f"  {name:16s} = {m:.4f} ± {s:.4f}")


    methods = []
    for d in per_seed_bl.values():
        for fam in d:
            for mth in d[fam]:
                if mth not in methods:
                    methods.append(mth)
    print("\n=== Open-set baselines: unknown-detection AUROC, mean ± std (all folds x seeds) ===")
    print("    (SVDD = ours; honest comparison against OpenMax / Energy / MSP)")
    method_summary = {}
    for mth in methods:
        vals = [d[fam][mth] for d in per_seed_bl.values() for fam in d if mth in d[fam]]
        m, s = _mean_std(vals)
        method_summary[mth] = (m, s, len(vals))
        print(f"  {mth:14s} {m:.4f} ± {s:.4f}   (n={len(vals)})")
    return per_seed, per_seed_bl, summary, method_summary



def run_ablation_multiseed(cfg, builder, seeds, held_family):
    print("\n" + "#" * 78)
    print(f"# ABLATION  x  MULTI-SEED   holdout='{held_family}'   seeds={list(seeds)}")
    print("#" * 78)
    per_seed = {}
    for sd in seeds:
        print(f"\n----- ablation seed {sd} -----")
        reseed(sd)
        train, val, test, _, _ = builder(cfg)
        per_seed[sd] = run_ablation(cfg, train, val, test, held_family=held_family)

    variants = list(next(iter(per_seed.values())).keys())
    print("\n=== Ablation: mean ± std across seeds (open-set holdout) ===")
    print(f"{'variant':16s} {'macro-F1':>17s} {'OSR-AUROC':>17s} {'unknown-recall':>18s}")
    out = {}
    for v in variants:
        f1 = [per_seed[s][v].get("known_f1_macro", float("nan")) for s in per_seed]
        au = [per_seed[s][v].get("osr_auroc", float("nan")) for s in per_seed]
        ur = [per_seed[s][v].get("unknown_recall", float("nan")) for s in per_seed]
        f1m, f1s = _mean_std(f1); aum, aus = _mean_std(au); urm, urs = _mean_std(ur)
        out[v] = dict(f1=(f1m, f1s), auroc=(aum, aus), unk_recall=(urm, urs))
        print(f"{v:16s} {f1m:7.4f} ± {f1s:6.4f} {aum:7.4f} ± {aus:6.4f} {urm:8.4f} ± {urs:6.4f}")
    return per_seed, out



@torch.no_grad()
def demo_forensic_llm_v2(model, test, label_map, cfg, feat_idx=None):
    print("\n=== LLM forensic use-case (ILLUSTRATIVE — prompt builder + sample report) ===")
    X, P, R, Y = test
    n = min(2048, X.size(0))
    x, p, r = X[:n].to(DEVICE), P[:n].to(DEVICE), R[:n].to(DEVICE)
    out = model.encode(x, p, r)
    dist, _ = model.svdd_dist(out["z"])
    dmin = dist.min(1).values.detach()
    j = int(dmin.argmax())
    xj, pj, rj = x[j:j + 1], p[j:j + 1], r[j:j + 1]
    logits = model.logits(model.encode(xj, pj, rj)["z"])
    target = logits.argmax(1)
    with torch.enable_grad():
        attr = integrated_gradients(model, xj, pj, rj, target)
    feat_rank = attr.abs().mean(1)[0]
    k = min(5, feat_rank.numel())
    top = feat_rank.argsort(descending=True)[:k].cpu().numpy()
    top_vals = feat_rank[top].cpu().numpy()
    phi = model.encode(xj, pj, rj)["phi"].detach().cpu().numpy()
    sev = "HIGH" if float(dmin[j].item()) > float(dmin.median().item()) else "MEDIUM"
    prompt = build_forensic_prompt(phi, float(dmin[j].item()), sev, top, top_vals,
                                   feat_global_idx=(feat_idx if feat_idx is not None else None))
    print(prompt)
    print("\n" + SAMPLE_LLM_REPORT)
    print("NOTE: wiring this prompt to a real LLM endpoint is left as deployment/future work;\n"
          "      no LLM output here is an evaluated contribution of the thesis.")
    return prompt



def main_full_v2(cfg=None):
    cfg = cfg or ConfigX()
    print("device:", DEVICE)
    print_documentation()

    if cfg.dataset == "unsw":
        builder = make_splits_unsw
        pred_builder = make_splits_unsw_pred
        held = "reconnaissance"
    else:
        set_taxonomy(["Normal", "DoS", "DDoS", "Probe", "Botnet",
                      "BruteForce", "WebAttack", "Infiltration"],
                     {"Normal": "benign", "DoS": "volumetric", "DDoS": "volumetric",
                      "Probe": "recon", "Botnet": "c2", "BruteForce": "credential",
                      "WebAttack": "webapp", "Infiltration": "intrusion"})
        builder = make_splits
        pred_builder = make_splits_pred
        held = "recon"

    reseed(cfg.seeds[0])
    train, val, test, feat_idx, scaler = builder(cfg)
    print(f"\ndataset={cfg.dataset}  classes={CLASS_NAMES}")
    print(f"sequences  train={tuple(train[0].shape)}  val={tuple(val[0].shape)}  test={tuple(test[0].shape)}")
    print(f"selected features ({len(feat_idx)}):", feat_idx.tolist())
    print("train label distribution:", torch.bincount(train[3], minlength=len(CLASS_NAMES)).tolist())

    known = set(range(len(CLASS_NAMES)))
    model, label_map, tau, res = run_pipeline(cfg, train, val, test, known)
    print("\n=== Closed-set baseline (all classes known) ===")
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in res.items()})
    demo_xai(model, test, label_map, cfg)
    operational_metrics(model, cfg, train[0].size(2))


    held_set = {i for i, c in enumerate(CLASS_NAMES) if ATTACK_FAMILY[c] == held}
    known_l = {i for i in range(len(CLASS_NAMES)) if i not in held_set}
    mdl_l, lm_l, tau_l, res_l = run_pipeline(cfg, train, val, test, known_l)
    print(f"\n=== Reference LOAFO fold (holdout '{held}') ===")
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in res_l.items()})
    threshold_sweep(mdl_l, test, lm_l, cfg, plot=False)
    gate_error_propagation(mdl_l, test, lm_l, tau_l, cfg)
    demo_forensic_llm_v2(mdl_l, test, lm_l, cfg, feat_idx)


    run_loafo_multiseed_v2(cfg, builder, cfg.seeds)


    run_prediction_demo(cfg, pred_builder)


    base_known = set(range(len(CLASS_NAMES)))
    if cfg.dataset == "unsw" and "Fuzzers" in CLASS_NAMES:
        base_known = base_known - {CLASS_NAMES.index("Fuzzers")}
    else:
        base_known = set(range(len(CLASS_NAMES) - 1))
    base_model, base_map, base_tau, base_res = run_pipeline(cfg, train, val, test, base_known)
    demo_continual_learning(cfg, base_model, base_known, train, val, test)


    run_ablation_multiseed(cfg, builder, cfg.seeds, held_family=held)


In [4]:
import builtins as _bi
import re as _re
import json as _json
from sklearn.metrics import confusion_matrix, matthews_corrcoef, cohen_kappa_score


@torch.no_grad()
def extended_known_metrics(model, test, label_map, tau, cfg):
    logits, dist, y_raw, ymap = gather_logits(model, test, label_map, cfg)
    probs = F.softmax(logits / model.temperature.cpu(), dim=1)
    keep = (ymap >= 0) & (dist <= tau)
    out = {"mcc": float("nan"), "kappa": float("nan"), "balanced_acc": float("nan")}
    if keep.any():
        true = ymap[keep].numpy()
        pred = probs[keep].argmax(1).numpy()
        if len(set(true.tolist())) > 1:
            out["mcc"] = float(matthews_corrcoef(true, pred))
            out["kappa"] = float(cohen_kappa_score(true, pred))
        from sklearn.metrics import balanced_accuracy_score
        out["balanced_acc"] = float(balanced_accuracy_score(true, pred))
    return out


def report_with_mcc(model, label_map, tau, test, cfg, title):
    base = evaluate(model, test, label_map, tau, cfg, set(label_map.keys()))
    ext = extended_known_metrics(model, test, label_map, tau, cfg)
    merged = {**base, **ext}
    print(f"\n=== {title} (incl. MCC / Kappa / balanced-acc) ===")
    keys = ["known_acc", "known_f1_macro", "mcc", "kappa", "balanced_acc",
            "ece", "osr_auroc", "unknown_recall"]
    print({k: (round(merged[k], 4) if isinstance(merged.get(k), float) else merged.get(k)) for k in keys})
    return merged


_DATASET_TRANSFER = {
    "unsw": dict(label="UNSW-NB15 (source)"),
}


def _align_features(Xsrc_dim, tensors, target_dim):
    X, P, R, Y = tensors
    f = X.size(2)
    if f == target_dim:
        return tensors
    if f > target_dim:
        X = X[:, :, :target_dim]
    else:
        pad = torch.zeros(X.size(0), X.size(1), target_dim - f)
        X = torch.cat([X, pad], dim=2)
    return (X, P, R, Y)


@torch.no_grad()
def zero_shot_transfer_eval(model, label_map, tau, cfg, ext_test, ext_class_names,
                            ext_normal_label=0, dataset_label="external"):
    print(f"\n=== Cross-dataset zero-shot transfer: {dataset_label} ===")
    raw_dim = model.embed.in_features if hasattr(model, "embed") else None
    Xt, Pt, Rt, Yt = ext_test
    if raw_dim is not None and Xt.size(2) != raw_dim:
        ext_test = _align_features(None, ext_test, raw_dim)
        Xt, Pt, Rt, Yt = ext_test
    Pt = Pt.clamp(0, cfg.n_port_buckets - 1)
    Rt = Rt.clamp(0, cfg.n_protocols - 1)
    dl = loader((Xt, Pt, Rt, Yt), cfg.batch_size, False)
    dists, attack_true = [], []
    for x, p, r, y in dl:
        x, p, r = x.to(DEVICE), p.to(DEVICE), r.to(DEVICE)
        out = model.encode(x, p, r)
        d, _ = model.svdd_dist(out["z"])
        dists.append(d.min(1).values.cpu())
        attack_true.append((y != ext_normal_label).long())
    dist = torch.cat(dists).numpy()
    yb = torch.cat(attack_true).numpy()
    res = {"dataset": dataset_label, "n": int(len(yb)), "attack_auroc": float("nan"),
           "attack_recall_at_tau": float("nan"), "fpr_at_tau": float("nan")}
    if 0 < yb.sum() < len(yb):
        res["attack_auroc"] = float(roc_auc_score(yb, dist))
        pred_attack = (dist > tau).astype(int)
        res["attack_recall_at_tau"] = float(recall_score(yb, pred_attack, zero_division=0))
        res["fpr_at_tau"] = float(pred_attack[yb == 0].mean()) if (yb == 0).any() else float("nan")
    print(f"  samples={res['n']}  attack-AUROC={res['attack_auroc']:.4f}  "
          f"recall@tau={res['attack_recall_at_tau']:.4f}  FPR@tau={res['fpr_at_tau']:.4f}")
    print("  note: source-trained model applied with NO retraining; an AUROC drop is expected and")
    print("        is reported honestly as a measure of cross-distribution (concept-drift) transfer.")
    return res


def build_external_from_unsw_split(cfg, perturb_std=0.35, seed=99):
    rng = np.random.RandomState(seed)
    if cfg.dataset == "unsw":
        _, _, test, _, _ = make_splits_unsw(cfg)
    else:
        _, _, test, _, _ = make_splits(cfg)
    X, P, R, Y = test
    Xn = X + torch.tensor(rng.normal(0, perturb_std, size=tuple(X.shape)), dtype=torch.float32)
    perm = list(range(X.size(2)))
    rng.shuffle(perm)
    keep = int(0.85 * X.size(2))
    mask = torch.zeros(X.size(2))
    mask[perm[:keep]] = 1.0
    Xn = Xn * mask.view(1, 1, -1)
    names = list(CLASS_NAMES)
    return (Xn, P, R, Y), names, 0, "UNSW-shifted (proxy external)"


def estimate_horizon_seconds(cfg):
    try:
        if cfg.dataset == "unsw":
            data = load_unsw_nb15(cfg)
        else:
            data = generate_flows(cfg)
    except Exception:
        return None
    ts = np.asarray(data["ts"], dtype=np.float64)
    host = np.asarray(data["host"]); svc = np.asarray(data["service"])
    deltas = []
    order = np.lexsort((ts, svc, host))
    h2 = host[order]; s2 = svc[order]; t2 = ts[order]
    for i in range(1, len(t2)):
        if h2[i] == h2[i - 1] and s2[i] == s2[i - 1]:
            dt = t2[i] - t2[i - 1]
            if 0 < dt < 1e6:
                deltas.append(dt)
    if not deltas:
        return None
    return float(np.median(deltas))


def horizon_to_time_report(cfg, horizons=(1, 2, 4)):
    print("\n=== Prediction horizon translated to wall-clock lead time ===")
    med = estimate_horizon_seconds(cfg)
    if med is None:
        print("  inter-flow timing unavailable; skipping wall-clock translation.")
        return {}
    out = {}
    for h in horizons:
        lead = med * h
        out[h] = lead
        print(f"  H={h}: median inter-flow gap {med:.3f}s  ->  lead time ≈ {lead:.3f}s ({lead/60:.2f} min)")
    print("  managerial reading: a one-step forecast yields the above early-warning window before the")
    print("  predicted flow materialises, directly widening the response window and lowering MTTR.")
    return out


def multi_horizon_prediction(cfg, pred_builder, horizons=(1, 2, 4)):
    print("\n" + "#" * 78)
    print(f"# MULTI-HORIZON PREDICTION  horizons={list(horizons)}")
    print("#" * 78)
    table = {}
    for h in horizons:
        cfgp = copy.deepcopy(cfg)
        cfgp.predict_horizon = h
        reseed(cfg.seeds[0] if hasattr(cfg, "seeds") else 1337)
        train, val, test, _, _ = pred_builder(cfgp)
        known = set(range(len(CLASS_NAMES)))
        model, label_map, tau, res = run_pipeline(cfgp, train, val, test, known)
        ews, ymap, normal_idx = early_warning_scores(model, test, label_map, cfgp)
        keep = ymap >= 0
        ews_auroc = float("nan")
        if normal_idx is not None:
            ya = (ymap[keep] != normal_idx).numpy().astype(int)
            if 0 < ya.sum() < len(ya):
                ews_auroc = float(roc_auc_score(ya, ews[keep].numpy()))
        table[h] = dict(acc=res.get("known_acc", float("nan")),
                        f1=res.get("known_f1_macro", float("nan")),
                        ews_auroc=ews_auroc)
        print(f"[H={h}] next-step acc={table[h]['acc']:.4f}  macro-F1={table[h]['f1']:.4f}  "
              f"EWS-AUROC={table[h]['ews_auroc']:.4f}")
    print("\n=== Horizon degradation summary ===")
    print(f"{'H':>3s} {'EWS-AUROC':>12s} {'next-step-acc':>14s}")
    for h in horizons:
        print(f"{h:>3d} {table[h]['ews_auroc']:>12.4f} {table[h]['acc']:>14.4f}")
    return table


def capture_loss_curves(cfg, train, val, test, known):
    pre, sup = [], []
    orig = _bi.print

    def cap(*a, **k):
        s = " ".join(str(x) for x in a)
        m = _re.search(r"\[pretrain\].*loss\s+([0-9.]+)", s)
        if m:
            pre.append(float(m.group(1)))
        m = _re.search(r"\[supervised\].*loss\s+([0-9.]+)", s)
        if m:
            sup.append(float(m.group(1)))
        orig(*a, **k)

    _bi.print = cap
    try:
        model, label_map, tau, res = run_pipeline(cfg, train, val, test, known)
    finally:
        _bi.print = orig
    return model, label_map, tau, res, pre, sup


@torch.no_grad()
def confusion_matrix_dump(model, test, label_map, tau, cfg):
    logits, dist, y_raw, ymap = gather_logits(model, test, label_map, cfg)
    probs = F.softmax(logits / model.temperature.cpu(), dim=1)
    inv = {v: k for k, v in label_map.items()}
    keep = (ymap >= 0) & (dist <= tau)
    if not keep.any():
        return [], []
    y_true = ymap[keep].numpy()
    y_pred = probs[keep].argmax(1).numpy()
    present = sorted(set(y_true.tolist()) | set(y_pred.tolist()))
    cm = confusion_matrix(y_true, y_pred, labels=present)
    names = [CLASS_NAMES[inv[i]] for i in present]
    return names, cm.tolist()


def emit_thesis_figures(cfg, builder, pred_builder):
    print("\n" + "#" * 78)
    print("# THESIS FIGURE EXPORTS  (confusion matrix, threshold sweep, loss curves)")
    print("#" * 78)
    reseed(cfg.seeds[0] if hasattr(cfg, "seeds") else 1337)
    train, val, test, feat_idx, scaler = builder(cfg)
    known_all = set(range(len(CLASS_NAMES)))
    model, label_map, tau, res, pre, sup = capture_loss_curves(cfg, train, val, test, known_all)

    cm_names, cm = confusion_matrix_dump(model, test, label_map, tau, cfg)
    print("\n==================== (1) CONFUSION MATRIX ====================")
    print("CM_LABELS =", _json.dumps(cm_names, ensure_ascii=False))
    print("CM =", _json.dumps(cm))

    held_family = "reconnaissance" if cfg.dataset == "unsw" else "recon"
    held = {i for i, c in enumerate(CLASS_NAMES) if ATTACK_FAMILY[c] == held_family}
    known_l = {i for i in range(len(CLASS_NAMES)) if i not in held}
    mdl_l, lm_l, tau_l, res_l = run_pipeline(cfg, train, val, test, known_l)
    rows = threshold_sweep(mdl_l, test, lm_l, cfg)
    sweep = [[float(q), float(t), float(fp), float(rc)] for (q, t, fp, rc) in rows]
    print("\n==================== (2) THRESHOLD SWEEP ====================")
    print("SWEEP_HOLDOUT =", held_family)
    print("SWEEP =", _json.dumps(sweep))

    print("\n==================== (3) LOSS CURVES ====================")
    print("LOSS_PRETRAIN =", _json.dumps([round(x, 4) for x in pre]))
    print("LOSS_SUPERVISED =", _json.dumps([round(x, 4) for x in sup]))
    print("\n# ---- copy the ==== blocks above into the chat to render the figures ----")
    return dict(cm_labels=cm_names, cm=cm, sweep=sweep, loss_pretrain=pre, loss_supervised=sup)


def main_full_v3(cfg=None):
    cfg = cfg or ConfigX()
    print("device:", DEVICE)
    print_documentation()

    if cfg.dataset == "unsw":
        builder = make_splits_unsw
        pred_builder = make_splits_unsw_pred
        held = "reconnaissance"
    else:
        set_taxonomy(["Normal", "DoS", "DDoS", "Probe", "Botnet",
                      "BruteForce", "WebAttack", "Infiltration"],
                     {"Normal": "benign", "DoS": "volumetric", "DDoS": "volumetric",
                      "Probe": "recon", "Botnet": "c2", "BruteForce": "credential",
                      "WebAttack": "webapp", "Infiltration": "intrusion"})
        builder = make_splits
        pred_builder = make_splits_pred
        held = "recon"

    reseed(cfg.seeds[0])
    train, val, test, feat_idx, scaler = builder(cfg)
    print(f"\ndataset={cfg.dataset}  classes={CLASS_NAMES}")
    print(f"sequences  train={tuple(train[0].shape)}  val={tuple(val[0].shape)}  test={tuple(test[0].shape)}")
    print(f"selected features ({len(feat_idx)}):", feat_idx.tolist())
    print("train label distribution:", torch.bincount(train[3], minlength=len(CLASS_NAMES)).tolist())

    known = set(range(len(CLASS_NAMES)))
    model, label_map, tau, res = run_pipeline(cfg, train, val, test, known)
    report_with_mcc(model, label_map, tau, test, cfg, "Closed-set baseline")
    demo_xai(model, test, label_map, cfg)
    operational_metrics(model, cfg, train[0].size(2))

    held_set = {i for i, c in enumerate(CLASS_NAMES) if ATTACK_FAMILY[c] == held}
    known_l = {i for i in range(len(CLASS_NAMES)) if i not in held_set}
    mdl_l, lm_l, tau_l, res_l = run_pipeline(cfg, train, val, test, known_l)
    report_with_mcc(mdl_l, lm_l, tau_l, test, cfg, f"Reference LOAFO fold (holdout '{held}')")
    threshold_sweep(mdl_l, test, lm_l, cfg, plot=False)
    gate_error_propagation(mdl_l, test, lm_l, tau_l, cfg)
    compare_osr_baselines(mdl_l, filter_known(train, known_l), test, lm_l, cfg)
    demo_forensic_llm_v2(mdl_l, test, lm_l, cfg, feat_idx)

    run_loafo_multiseed_v2(cfg, builder, cfg.seeds)
    run_ablation_multiseed(cfg, builder, cfg.seeds, held_family=held)

    horizon_to_time_report(cfg, horizons=(1, 2, 4))
    multi_horizon_prediction(cfg, pred_builder, horizons=(1, 2, 4))

    ext_test, ext_names, ext_norm, ext_label = build_external_from_unsw_split(cfg)
    zero_shot_transfer_eval(model, label_map, tau, cfg, ext_test, ext_names, ext_norm, ext_label)

    base_known = set(range(len(CLASS_NAMES)))
    if cfg.dataset == "unsw" and "Fuzzers" in CLASS_NAMES:
        base_known = base_known - {CLASS_NAMES.index("Fuzzers")}
    else:
        base_known = set(range(len(CLASS_NAMES) - 1))
    base_model, base_map, base_tau, base_res = run_pipeline(cfg, train, val, test, base_known)
    demo_continual_learning(cfg, base_model, base_known, train, val, test)
    print("\nNOTE on continual learning: results are reported as CONDITIONAL — replay+EWC prevents total")
    print("collapse (overall accuracy retained) but minority-class macro-F1 still drops; abstract/claims")
    print("must say 'mitigates' not 'prevents' catastrophic forgetting.")

    emit_thesis_figures(cfg, builder, pred_builder)


if __name__ == "__main__":
    main_full_v3()


device: cuda


CORE CONTRIBUTION (single, foregrounded):
    Threat-Context-Conditioned Adaptive Fusion.
    A per-dimension (vector) gate that fuses the Transformer and BiLSTM branches,
    *conditioned on a learnable Threat-Context vector phi_c*, where the SAME phi_c
    also drives the curriculum masking strategy of the self-supervised pretext task.
    One mechanism, used at two points (fusion gate + mask guidance) — this is the
    novel contribution around which the whole architecture is organised.

SUPPORTING COMPONENTS (explicitly NOT claimed as standalone novelty):
    - Self-supervised pretraining (MTM curriculum + semantic-preserving contrastive)
    - Class-conditional Deep-SVDD open-world gate with anti-collapse + quantile threshold
    - Continual-learning loop (Replay + EWC), analyst-in-the-loop
    - Operational Trust Layer  (renamed from "Explainability"):
        Attention-Rollout (online) + Integrated-Gradients (offline) + Temperature
        calibration, validated b

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



=== Reference LOAFO fold (holdout 'reconnaissance') (incl. MCC / Kappa / balanced-acc) ===
{'known_acc': 0.9973, 'known_f1_macro': 0.4515, 'mcc': 0.9839, 'kappa': 0.9839, 'balanced_acc': 0.6105, 'ece': 0.0184, 'osr_auroc': np.float64(0.9738), 'unknown_recall': 0.9574}

=== Threshold sweep (false-alarm vs unknown-recall) ===
  quantile   tau      FPR(false-alarm)  unknown-recall
   0.500   0.0175      0.4999           1.0000
   0.562   0.0185      0.4376           1.0000
   0.625   0.0195      0.3751           1.0000
   0.687   0.0203      0.3129           1.0000
   0.750   0.0214      0.2505           1.0000
   0.812   0.0230      0.1882           1.0000
   0.874   0.0257      0.1258           1.0000
   0.937   0.0515      0.0634           1.0000
   0.999   3.1399      0.0010           0.0000

=== Gate error-propagation analysis ===
  gate_fpr_known_to_unknown        = 0.0503
  gate_fnr_unknown_to_known        = 0.0426
  end_to_end_acc                   = 0.9472
  oracle_gate_acc     

/usr/local/lib/python3.12/dist-packages/scipy/stats/_continuous_distns.py:2756: RuntimeWarning: overflow encountered in power
  return -sc.expm1(-pow(x, c))


  SVDD (ours)    AUROC = 0.9696
  Energy-OOD     AUROC = 0.9781
  MSP            AUROC = 0.9767
  OpenMax        AUROC = 0.7106
[LOAFO] dos              auroc=0.9699  unk_recall=0.8878  known_F1=0.4012
[pretrain] epoch 1/6 loss 3.6832
[pretrain] epoch 2/6 loss 3.1810
[pretrain] epoch 3/6 loss 2.8663
[pretrain] epoch 4/6 loss 2.7416
[pretrain] epoch 5/6 loss 2.7085
[pretrain] epoch 6/6 loss 2.6897
[supervised] epoch 1/12 loss 0.2888
[supervised] epoch 2/12 loss 0.1907
[supervised] epoch 3/12 loss 0.1664
[supervised] epoch 4/12 loss 0.1494
[supervised] epoch 5/12 loss 0.1371
[supervised] epoch 6/12 loss 0.1314
[supervised] epoch 7/12 loss 0.1202
[supervised] epoch 8/12 loss 0.1111
[supervised] epoch 9/12 loss 0.1032
[supervised] epoch 10/12 loss 0.1021
[supervised] epoch 11/12 loss 0.0977
[supervised] epoch 12/12 loss 0.0953
[calibration] temperature 0.810  threshold tau 0.020

=== Open-set baselines (unknown-detection AUROC, same fold) ===
  SVDD (ours)    AUROC = 0.9764
  Energy-OOD   

100%|██████████| 149M/149M [00:01<00:00, 98.8MB/s]

Extracting files...


[pretrain] epoch 1/6 loss 5.0696
[pretrain] epoch 2/6 loss 3.8016
[pretrain] epoch 3/6 loss 3.7231
[pretrain] epoch 4/6 loss 3.5740
[pretrain] epoch 5/6 loss 3.3894
[pretrain] epoch 6/6 loss 3.3318
[supervised] epoch 1/12 loss 0.3665
[supervised] epoch 2/12 loss 0.2334
[supervised] epoch 3/12 loss 0.2107
[supervised] epoch 4/12 loss 0.1975
[supervised] epoch 5/12 loss 0.1837
[supervised] epoch 6/12 loss 0.1736
[supervised] epoch 7/12 loss 0.1631
[supervised] epoch 8/12 loss 0.1580
[supervised] epoch 9/12 loss 0.1486
[supervised] epoch 10/12 loss 0.1375
[supervised] epoch 11/12 loss 0.1355
[supervised] epoch 12/12 loss 0.1328
[calibration] temperature 0.936  threshold tau 0.213
[H=2] next-step acc=0.9918  macro-F1=0.4193  EWS-AUROC=0.9941
Using Colab cache for faster access to the 'unsw-nb15' dataset.
[pretrain] epoch 1/6 loss 11.1823
[pretrain] epoch 2/6 loss 10.2708
[pretrain] epoch 3/6 loss 8.1879
[pretrain] epoch 4/6 loss 4.8635
[pretrain] epoch 5/6 loss 4.7448
[pretrain] epoch 6/6 